# §4.5 LLM Experiment — Nomadic Signal Transfer (Task-Domain LoRA)

**실험 목적**: Nomadic Intelligence의 core signal layer가 대형 autoregressive LLM에
이식 가능한지 검증. 특히 task-domain 기반 12-Expert LoRA switching에서
ΔH (entropy differentiation) signature가 나타나는지 확인.

**기존 §4.5 (Gemma-4-E2B) 대비 개선점**:
- **Expert 12개** (math×4 / code×4 / language×4) — sub-specialization 도입
- **대조군 5종 추가**: MoD, ECR, Sigma-MoE, Bi-Level MoE, Routing-Free MoE
- **7가지 스트레스 시나리오**: 작전 1~7 (도메인급변 / 노이즈면역 / 지속학습 /
  중첩경계 / 유혹회복 / 정보획득효율 / 장기고착도)
- **확장 지표**: Expert Load & Diversity / Transition Sharpness / Latency & FLOPs /
  Recovery Steps / corr(Δx, NLL) / Convergence Step / Decay Rate

**시나리오 전체 목록**:
```
A. 일반 도메인 평가       (math / code / language eval prompts)
B. 도메인 급변            (일상 → 수학 텍스트 shift, Δx 시계열)
C. 노이즈 면역력          (clean vs 15% 오타 주입 텍스트)
D. 중첩 경계              (math×code×language 복합 프롬프트, 평형 vs 진동)
E. 유혹과 회복            (수학 중 일상 농담 주입, 원래 expert 복귀 속도)
F. 정보 획득 효율성       (고밀도 vs 저밀도, corr(Δx, NLL))
G. 장기 문맥 고착도       (단일 도메인 600토큰, Δx → 0 수렴 확인)
```

**모델 선택** (STEP 0에서 변경):
```
MODEL_KEY = 'gemma4_26b'    # Gemma-4-26B-A4B-base (MoE)
MODEL_KEY = 'mistral_24b'   # Mistral-Small-24B-base
MODEL_KEY = 'llama_8b'      # Meta-Llama-3.1-8B-base
```

**비교 모델**: Nomadic Full / MoD / ECR / Sigma-MoE / Bi-Level-MoE / Routing-Free-MoE  
**GPU**: Colab Pro+ (A100) | 4-bit NF4 quantization

In [ ]:
# ============================================================
# STEP 0: 드라이브 마운트 + 경로 설정
# ============================================================
from google.colab import drive
import os, json, time
drive.mount('/content/drive', force_remount=True)

DRIVE_BASE  = '/content/drive/MyDrive/nomadic_llm_results'
MODELS_BASE = '/content/drive/MyDrive/models'
RUN_ID  = time.strftime('%Y%m%d_%H%M%S')

# ── 모델 선택 ─────────────────────────────────────────────
MODEL_KEY = 'gemma4_26b'   # ← 여기만 변경

MODEL_REGISTRY = {
    'gemma4_26b':    os.path.join(MODELS_BASE, 'Gemma-4-26B-a4b-base'),
    'gemma4_26b_it': os.path.join(MODELS_BASE, 'gemma-4-26B-A4B-it'),
    'mistral_24b':   os.path.join(MODELS_BASE, 'mistral-small-24b-base'),
    'exaone_33b':    os.path.join(MODELS_BASE, 'EXAONE-4.5B-33B-base'),
    'llama_8b':      os.path.join(MODELS_BASE, 'Meta-Llama-3.1-8B-base'),
}
MODEL_PATH = MODEL_REGISTRY[MODEL_KEY]
# ──────────────────────────────────────────────────────────

RUN_DIR = os.path.join(DRIVE_BASE, f'{MODEL_KEY}_{RUN_ID}')
os.makedirs(RUN_DIR, exist_ok=True)

print(f'Model: {MODEL_KEY}')
print(f'Path:  {MODEL_PATH}')
print(f'Save:  {RUN_DIR}')

Mounted at /content/drive
Model: gemma4_26b
Path:  /content/drive/MyDrive/models/Gemma-4-26B-a4b-base
Save:  /content/drive/MyDrive/nomadic_llm_results/gemma4_26b_20260419_021120


In [ ]:
# ============================================================
# STEP 1: 패키지 설치 + 임포트
# ============================================================
!pip install -q transformers accelerate bitsandbytes peft

import warnings
warnings.filterwarnings('ignore')

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import deque, defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, TaskType, PeftModel
from torch.optim import AdamW

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

CUDA: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


In [ ]:
import os
!pip install --upgrade transformers
# ============================================================
# STEP 2: 모델 로드 (4-bit NF4)
# ============================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f'{MODEL_KEY} 로드 중...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
base_model.eval()

# hidden_dim 확인
dummy = tokenizer('test', return_tensors='pt').input_ids.to(base_model.device)
with torch.no_grad():
    out = base_model(dummy, output_hidden_states=True)
HIDDEN_DIM = out.hidden_states[-1].shape[-1]
print(f'✅ 로드 완료 | hidden_dim={HIDDEN_DIM}')

gemma4_26b 로드 중...


Loading weights:   0%|          | 0/1013 [00:00<?, ?it/s]

✅ 로드 완료 | hidden_dim=2816


In [ ]:
# ============================================================
# STEP 3: Nomadic 컴포넌트 정의
# ============================================================
class HybridDeltaTracker:
    """기존 §4.5와 동일한 구조. alpha/beta = ema_decay/baseline_momentum."""
    def __init__(self, alpha=0.8, beta=0.85,
                 tau_min=2.0, tau_max=8.0,
                 tau_var_scale=6.0, tau_window=8):
        self.alpha = alpha; self.beta = beta
        self.tau_min = tau_min; self.tau_max = tau_max
        self.tau_var_scale = tau_var_scale; self.tau_window = tau_window
        self.reset()

    def reset(self):
        self.prev_x = None
        self.ema_err = 0.0; self.baseline_err = 0.0
        self.recent_de = deque(maxlen=self.tau_window)
        self.history = []

    def compute(self, current_x, current_err):
        if self.prev_x is None:
            delta_env = 0.0
        else:
            cos = F.cosine_similarity(
                current_x.float().flatten(),
                self.prev_x.float().flatten(), dim=0)
            delta_env = float(1.0 - cos.item())
        self.prev_x = current_x.detach().clone()
        self.recent_de.append(delta_env)

        self.ema_err      = self.alpha * self.ema_err + (1-self.alpha) * current_err
        self.baseline_err = self.beta  * self.baseline_err + (1-self.beta) * current_err
        delta_err = max(0.0, self.ema_err - self.baseline_err)

        delta_hybrid = float(torch.tanh(torch.tensor(delta_env + delta_err)).item())
        sigma2 = float(np.var(self.recent_de)) if len(self.recent_de) >= 2 else 0.0
        tau_dyn = self.tau_min + (self.tau_max - self.tau_min) / (1.0 + self.tau_var_scale * sigma2)

        rec = dict(delta_env=delta_env, delta_err=delta_err,
                   delta_hybrid=delta_hybrid, sigma2=sigma2, tau_dyn=tau_dyn)
        self.history.append(rec)
        return rec


class NomadicPolicyNet(nn.Module):
    """
    12-Expert 버전.
    num_experts=12: math_0..3 / code_0..3 / language_0..3
    """
    def __init__(self, hidden_dim, policy_hidden=128, num_experts=12):
        super().__init__()
        self.proj = nn.Linear(hidden_dim, policy_hidden)
        self.shared = nn.Sequential(
            nn.Linear(policy_hidden + 5, policy_hidden), nn.ReLU(),
            nn.Linear(policy_hidden, policy_hidden), nn.ReLU(),
        )
        self.stay_switch_head = nn.Linear(policy_hidden, 2)
        self.target_head      = nn.Linear(policy_hidden, num_experts)
        self.mode_head        = nn.Linear(policy_hidden, 2)

    def forward(self, hidden, meta_signals):
        if hidden.dim() == 3: hidden = hidden.squeeze(1)
        h = F.relu(self.proj(hidden.float()))
        inp = torch.cat([h, meta_signals.float()], dim=-1)
        h = self.shared(inp)
        return (F.softmax(self.stay_switch_head(h), dim=-1),
                F.softmax(self.target_head(h),      dim=-1),
                F.softmax(self.mode_head(h),        dim=-1))


def build_meta_signals(rec, device):
    return torch.tensor([[
        rec['delta_hybrid'],
        rec['delta_err'],
        rec['delta_hybrid'],
        math.tanh(rec['sigma2'] * 10.0),
        math.tanh((rec['tau_dyn'] - 5.0) / 5.0),
    ]], dtype=torch.float32, device=device)


def topk_entropy(logits, k=50):
    probs = F.softmax(logits, dim=-1)
    topk  = probs.topk(k).values
    topk  = topk / topk.sum()
    return float(-(topk * (topk + 1e-10).log()).sum().item())


# ─── 새 지표 유틸리티 ───────────────────────────────────────

def expert_load_stats(expert_visit_counts: dict) -> dict:
    """
    Expert Load & Diversity 지표.
    - load_cv   : Coefficient of Variation (낮을수록 균등 분배)
    - load_entropy : Expert 방문 분포의 엔트로피 (높을수록 다양)
    - gini      : Gini 계수 (0=완전균등, 1=완전집중)
    expert_visit_counts: {expert_name: visit_count, ...}
    """
    counts = np.array(list(expert_visit_counts.values()), dtype=float)
    total  = counts.sum()
    if total == 0:
        return dict(load_cv=0.0, load_entropy=0.0, gini=0.0,
                    max_load=0.0, min_load=0.0)
    probs  = counts / total
    cv     = float(counts.std() / (counts.mean() + 1e-9))
    H_exp  = float(-(probs * np.log(probs + 1e-10)).sum())
    # Gini
    s = counts.copy(); s.sort()
    n = len(s)
    gini = float(1 - 2 * (np.cumsum(s) / (s.sum() + 1e-9) * (1/n)).sum())
    return dict(load_cv=cv, load_entropy=H_exp, gini=gini,
                max_load=float(probs.max()), min_load=float(probs.min()))


def transition_sharpness(expert_trace: list) -> dict:
    """
    Transition Sharpness / Settling Time 지표.
    expert_trace: [expert_name_per_step, ...]
    - settling_time : 전환 후 동일 expert 연속 체류 스텝 수의 평균
    - switch_rate   : 스텝당 전환 횟수
    - oscillation   : 연속 두 번 전환(A→B→A) 발생 비율
    """
    if len(expert_trace) < 2:
        return dict(settling_time=0.0, switch_rate=0.0, oscillation=0.0)
    switches = [i for i in range(1, len(expert_trace))
                if expert_trace[i] != expert_trace[i-1]]
    switch_rate = len(switches) / len(expert_trace)

    # settling time: 각 전환 후 다음 전환까지 스텝 수
    settling_times = []
    for j, sw in enumerate(switches):
        next_sw = switches[j+1] if j+1 < len(switches) else len(expert_trace)
        settling_times.append(next_sw - sw)
    settling_time = float(np.mean(settling_times)) if settling_times else float(len(expert_trace))

    # oscillation A→B→A
    oscillations = 0
    for k in range(2, len(expert_trace)):
        if expert_trace[k] == expert_trace[k-2] and expert_trace[k] != expert_trace[k-1]:
            oscillations += 1
    oscillation = oscillations / max(1, len(expert_trace) - 2)

    return dict(settling_time=settling_time, switch_rate=switch_rate,
                oscillation=oscillation)


def measure_latency_flops(gen_fn, prompt, n_warmup=1, n_measure=3):
    """
    Inference Latency & 추정 FLOPs 지표.
    - latency_ms : 토큰당 평균 레이턴시 (밀리초)
    - tokens_per_sec : 초당 생성 토큰 수
    FLOPs는 토큰 수 × 모델 파라미터 수 × 2 (MAC 근사) 로 추정.
    """
    import time
    # warmup
    for _ in range(n_warmup):
        gen_fn(prompt)
    # measure
    times, n_tokens = [], []
    for _ in range(n_measure):
        t0 = time.perf_counter()
        res = gen_fn(prompt)
        t1 = time.perf_counter()
        n_tok = len(tokenizer.encode(res['text'])) if 'text' in res else MAX_NEW_TOKENS
        times.append(t1 - t0)
        n_tokens.append(n_tok)
    avg_time = float(np.mean(times))
    avg_toks = float(np.mean(n_tokens))
    latency_ms = avg_time / max(1, avg_toks) * 1000.0
    tps = avg_toks / (avg_time + 1e-9)
    # 파라미터 기반 FLOPs 근사
    n_params = sum(p.numel() for p in base_model.parameters())
    est_flops = 2 * n_params * avg_toks  # MAC × 2
    return dict(latency_ms=latency_ms, tokens_per_sec=tps,
                est_flops=est_flops, avg_tokens=avg_toks)


print('✅ Nomadic 컴포넌트 정의 완료 (12-Expert, 확장 지표 포함)')


✅ Nomadic 컴포넌트 정의 완료 (12-Expert, 확장 지표 포함)


In [ ]:
# ============================================================
# STEP 4: Task-Domain 프롬프트 정의
#
# 기존 §4.5: stable/transition/creative (heuristic)
# 이번 실험: math/code/language (task-domain)
#
# 세 domain은 distributional boundary가 명확:
#   Math   → 고확신, 정해진 답, 낮은 불확실성
#   Code   → 구조적, 문법 제약, 중간 불확실성
#   Language → 탐색적, 자유 생성, 높은 불확실성
# ============================================================

# PolicyNet 학습용 프롬프트
MATH_PROMPTS_TRAIN = [
    "2 더하기 2는 4이다. 3 더하기 5는",
    "피타고라스 정리: 직각삼각형에서 빗변의 제곱은",
    "1에서 10까지의 합은 55이다. 1에서 5까지의 합은",
    "The derivative of x^2 is 2x. The derivative of x^3 is",
    "If a = 3 and b = 4, then a + b =",
    "소수(prime number)의 정의: 1과 자기 자신으로만 나누어지는 수. 가장 작은 소수는",
    "삼각형의 내각의 합은 180도이다. 사각형의 내각의 합은",
    "log(100) = 2이다. log(1000) =",
]

CODE_PROMPTS_TRAIN = [
    "def factorial(n):\n    if n == 0:\n        return 1\n    return n *",
    "# Python list comprehension\nresult = [x**2 for x in range(10)]\n# 결과:",
    "import numpy as np\narr = np.array([1, 2, 3])\nprint(arr.mean()  #",
    "function add(a, b) {\n  return a + b;\n}\nconsole.log(add(3, 4));  /",
    "SELECT * FROM users WHERE age >",
    "class Animal:\n    def __init__(self, name):\n        self.name =",
    "for i in range(5):\n    print(i)\n# 출력:",
    "git commit -m \"Initial commit\"\ngit push origin",
]

LANGUAGE_PROMPTS_TRAIN = [
    "처음에는 모든 것이 안정적이었다. 그러나 갑자기",
    "The ancient philosopher once said that the nature of reality is",
    "달빛이 창문을 통해 들어오던 그날 밤, 그는",
    "What if consciousness is not a product of the brain but",
    "미래의 도시는 어떤 모습일까? 어쩌면",
    "In a world where time flows backwards,",
    "그 순간 그녀는 모든 것이 변했다는 것을 알았다. 왜냐하면",
    "The most beautiful thing about uncertainty is",
]

# 평가용 프롬프트 (학습과 분리)
MATH_PROMPTS_EVAL = [
    "24를 6으로 나누면",
    "The square root of 144 is",
    "If x + 5 = 12, then x =",
    "원의 넓이 공식은 πr²이다. 반지름이 3이면 넓이는",
    "등차수열 2, 5, 8, 11의 다음 항은",
]

CODE_PROMPTS_EVAL = [
    "# 피보나치 수열\ndef fib(n):\n    if n <= 1: return n\n    return",
    "arr = [3, 1, 4, 1, 5]\narr.sort()\nprint(arr)  #",
    "const greeting = (name) => `Hello,",
    "try:\n    result = 10 / 0\nexcept ZeroDivisionError:",
    "<html>\n<body>\n<h1>Hello</h1>\n</body>\n</",
]

LANGUAGE_PROMPTS_EVAL = [
    "바람이 부는 날, 그는 오래된 편지를",
    "The paradox of freedom is that",
    "만약 내가 다시 태어난다면",
    "In the silence between words, there is",
    "존재한다는 것의 의미를 묻는다면",
]

# domain → (학습용, 평가용, switch_label)
# math/code: 예측 가능 → stay(0), hard mode(1)
# language: 탐색적 → switch(1), soft mode(0)
DOMAIN_CONFIG = {
    'math':     (MATH_PROMPTS_TRAIN,     MATH_PROMPTS_EVAL,     0, 1),
    'code':     (CODE_PROMPTS_TRAIN,     CODE_PROMPTS_EVAL,     0, 1),
    'language': (LANGUAGE_PROMPTS_TRAIN, LANGUAGE_PROMPTS_EVAL, 1, 0),
}

print(f'Train prompts: math={len(MATH_PROMPTS_TRAIN)}, '
      f'code={len(CODE_PROMPTS_TRAIN)}, '
      f'language={len(LANGUAGE_PROMPTS_TRAIN)}')
print(f'Eval prompts:  math={len(MATH_PROMPTS_EVAL)}, '
      f'code={len(CODE_PROMPTS_EVAL)}, '
      f'language={len(LANGUAGE_PROMPTS_EVAL)}')

Train prompts: math=8, code=8, language=8
Eval prompts:  math=5, code=5, language=5


In [ ]:
# ============================================================
# STEP 5: LoRA Expert 학습 — 12개 Sub-Expert
#
# 도메인 3 × Sub-Expert 4 = 12개
#   math_0  : 기초 산술/대수          math_1  : 미적분/해석학
#   math_2  : 이산수학/논리            math_3  : 통계/확률
#   code_0  : Python/알고리즘         code_1  : JS/웹
#   code_2  : SQL/데이터               code_3  : 시스템/CLI
#   language_0 : 서사/소설             language_1 : 철학/사변
#   language_2 : 한국어 감성           language_3 : 영어 논증
#
# phase4(E2B) 대비 개선:
#   - (prompt, completion) 쌍으로 학습: completion 부분만 loss 계산
#   - sub-domain별 특화 데이터 제공
# ============================================================
from peft.tuners.lora import LoraLayer
from peft import prepare_model_for_kbit_training

LORA_R       = 4   # T4: 4, A100: 8
LORA_ALPHA   = 8
LORA_DROPOUT = 0.05
LORA_EPOCHS  = 3

# 12-Expert 목록 (도메인 × sub_id)
EXPERT_KEYS = [
    'math_0', 'math_1', 'math_2', 'math_3',
    'code_0', 'code_1', 'code_2', 'code_3',
    'language_0', 'language_1', 'language_2', 'language_3',
]
NUM_EXPERTS = len(EXPERT_KEYS)  # 12

# 도메인 그룹 매핑 (라우팅 fallback용)
EXPERT_DOMAIN = {k: k.split('_')[0] for k in EXPERT_KEYS}

# Expert별 (prompt, completion) 학습 데이터
EXPERT_DATA = {
    # ── Math ──────────────────────────────────────────────
    'math_0': [  # 기초 산술/대수
        ("2 더하기 2는 4이다. 3 더하기 5는",          "8이다."),
        ("If a = 3 and b = 4, then a + b =",           "7."),
        ("삼각형의 내각의 합은 180도이다. 사각형의 내각의 합은", "360도이다."),
        ("The square root of 144 is",                   "12."),
        ("log(100) = 2이다. log(1000) =",               "3이다."),
        ("등차수열 2, 5, 8, 11의 다음 항은",            "14이다."),
    ],
    'math_1': [  # 미적분/해석학
        ("The derivative of x^2 is 2x. The derivative of x^3 is", "3x^2."),
        ("The integral of 2x dx is",                    "x^2 + C."),
        ("lim(x→0) sin(x)/x =",                        "1."),
        ("The chain rule states: d/dx[f(g(x))] =",     "f'(g(x)) · g'(x)."),
        ("If f(x) = e^x, then f'(x) =",               "e^x."),
        ("피타고라스 정리: 직각삼각형에서 빗변의 제곱은", "나머지 두 변의 제곱의 합과 같다."),
    ],
    'math_2': [  # 이산수학/논리
        ("소수(prime number)의 정의: 1과 자기 자신으로만 나누어지는 수. 가장 작은 소수는", "2이다."),
        ("집합 A={1,2,3}과 B={2,3,4}의 교집합은",       "{2,3}이다."),
        ("명제 P→Q의 대우는",                            "¬Q→¬P이다."),
        ("n=5일 때 n!은",                               "120이다."),
        ("이진수 1010을 십진수로 변환하면",               "10이다."),
        ("그래프 이론에서 완전 그래프 K4의 간선 수는",   "6개이다."),
    ],
    'math_3': [  # 통계/확률
        ("동전을 두 번 던질 때 앞면이 두 번 나올 확률은", "1/4이다."),
        ("정규분포에서 평균±1σ 범위에 포함되는 데이터 비율은 약", "68%이다."),
        ("1, 2, 3, 4, 5의 평균은",                     "3이다."),
        ("두 독립 사건 A, B에서 P(A∩B) =",             "P(A) × P(B)이다."),
        ("중앙값과 평균이 다를 때 분포는",               "비대칭(skewed)임을 시사한다."),
        ("표준편차는 분산의",                            "제곱근이다."),
    ],
    # ── Code ──────────────────────────────────────────────
    'code_0': [  # Python/알고리즘
        ("def factorial(n):\n    if n == 0:\n        return 1\n    return n *",
         "factorial(n - 1)"),
        ("# Python list comprehension\nresult = [x**2 for x in range(5)]\n# 결과:",
         "[0, 1, 4, 9, 16]"),
        ("class Animal:\n    def __init__(self, name):\n        self.name =",
         "name"),
        ("def binary_search(arr, target):\n    lo, hi = 0, len(arr)-1\n    while lo <= hi:\n        mid =",
         "(lo + hi) // 2"),
        ("# 리스트를 뒤집는 파이썬 슬라이싱\nreversed_list = my_list",
         "[::-1]"),
        ("try:\n    result = 10 / 0\nexcept ZeroDivisionError:",
         "\n    print(\"Cannot divide by zero\")"),
    ],
    'code_1': [  # JS/웹
        ("function add(a, b) {\n  return a + b;\n}\nconsole.log(add(3, 4));  //",
         "7"),
        ("const greeting = (name) => `Hello,",
         "${name}!`"),
        ("document.querySelector('#btn').addEventListener('click', () => {",
         "\n  console.log('clicked');\n});"),
        ("fetch('/api/data')\n  .then(res => res.json())\n  .then(data =>",
         "console.log(data)"),
        ("<html>\n<body>\n<h1>Hello</h1>\n</body>\n</",
         "html>"),
        ("const [count, setCount] = useState(0);\n// increment:\n",
         "setCount(count + 1);"),
    ],
    'code_2': [  # SQL/데이터
        ("SELECT * FROM users WHERE age >",             "18 ORDER BY name ASC;"),
        ("SELECT COUNT(*) FROM orders WHERE status =",  "'completed';"),
        ("import pandas as pd\ndf = pd.read_csv('data.csv')\nprint(df.head(",
         "5))"),
        ("df.groupby('category')['sales'].sum()",       ".sort_values(ascending=False)"),
        ("SELECT u.name, COUNT(o.id) FROM users u\nLEFT JOIN orders o ON u.id = o.user_id\nGROUP BY",
         "u.name;"),
        ("df[df['age'] > 30].reset_index(",             "drop=True)"),
    ],
    'code_3': [  # 시스템/CLI
        ("git commit -m \"Initial commit\"\ngit push origin",
         "main"),
        ("ls -la /home/user | grep",                    '\".py\"'),
        ("docker run -d -p 8080:80",                    "--name myapp nginx"),
        ("chmod +x script.sh\n./",                     "script.sh"),
        ("ps aux | grep python | awk '{print $2}' | xargs",
         "kill"),
        ("curl -X POST https://api.example.com/data -H 'Content-Type: application/json' -d",
     '\'{"key": "value"}\'' ), # 바깥쪽을 작은따옴표로, 안쪽 제이슨은 큰따옴표로 유지
    ],
    # ── Language ───────────────────────────────────────────
    'language_0': [  # 서사/소설
        ("처음에는 모든 것이 안정적이었다. 그러나 갑자기",
         "예상치 못한 일이 벌어지기 시작했다."),
        ("달빛이 창문을 통해 들어오던 그날 밤, 그는",
         "오래된 기억 속으로 빠져들었다."),
        ("그 순간 그녀는 모든 것이 변했다는 것을 알았다. 왜냐하면",
         "더 이상 두려움이 느껴지지 않았기 때문이다."),
        ("In a world where time flows backwards,",
         "every farewell becomes a first meeting."),
        ("The old lighthouse keeper had not spoken to anyone for",
         "thirty years, but the sea always listened."),
        ("바람이 부는 날, 그는 오래된 편지를",
         "서랍 깊은 곳에서 꺼내 펼쳤다."),
    ],
    'language_1': [  # 철학/사변
        ("What if consciousness is not a product of the brain but",
         "a fundamental property of the universe itself?"),
        ("The most beautiful thing about uncertainty is",
         "that it leaves room for possibility."),
        ("In the silence between words, there is",
         "a language older than speech."),
        ("존재한다는 것의 의미를 묻는다면",
         "우리는 결국 자기 자신에게 돌아오게 된다."),
        ("The paradox of freedom is that",
         "it requires constraints to be meaningful."),
        ("만약 내가 다시 태어난다면",
         "같은 선택을 할지 확신할 수 없다."),
    ],
    'language_2': [  # 한국어 감성
        ("미래의 도시는 어떤 모습일까? 어쩌면",
         "인간과 AI가 함께 설계하는 공간이 될 것이다."),
        ("가을 낙엽이 떨어질 때마다 그는",
         "지나간 시간들을 하나씩 떠올렸다."),
        ("어머니의 손길은 언제나",
         "따뜻하고 익숙한 냄새가 났다."),
        ("첫사랑의 기억은 왜 이토록",
         "선명하게 남아 있는 걸까."),
        ("서울의 밤거리를 걸으며 그는",
         "이 도시가 여전히 낯설게 느껴졌다."),
        ("비가 내리는 날엔 어쩐지",
         "모든 것이 조금 더 솔직해지는 것 같다."),
    ],
    'language_3': [  # 영어 논증
        ("The ancient philosopher once said that the nature of reality is",
         "beyond what the senses can perceive."),
        ("Although science seemed to have all the answers, suddenly",
         "a new question emerged that challenged everything."),
        ("The evidence suggests that early intervention is more effective because",
         "it addresses root causes rather than symptoms."),
        ("Critics argue that this policy fails to account for",
         "the long-term consequences on future generations."),
        ("To summarize, the data consistently shows that",
         "correlation does not imply causation."),
        ("One must consider the ethical implications before",
         "proceeding with any irreversible action."),
    ],
}

# DOMAIN_CONFIG의 학습 프롬프트도 sub-expert로 분배
# (collect_training_data에서 참조)
DOMAIN_TO_EXPERTS = {
    'math':     ['math_0', 'math_1', 'math_2', 'math_3'],
    'code':     ['code_0', 'code_1', 'code_2', 'code_3'],
    'language': ['language_0', 'language_1', 'language_2', 'language_3'],
}

def get_lora_config():
    return LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=['q_proj.linear', 'v_proj.linear', 'k_proj.linear', 'o_proj.linear'],
        bias='none',
    )

def train_lora_expert(expert_key, expert_pairs, save_dir):
    """
    (prompt, completion) 쌍으로 LoRA expert 학습.
    completion 부분만 loss 계산 (prompt 토큰 = -100 마스킹).
    """
    save_path = os.path.join(save_dir, f'expert_{expert_key}')
    if os.path.exists(save_path):
        print(f'  [{expert_key}] 이미 학습됨 — 스킵')
        return save_path

    print(f'  [{expert_key}] LoRA 학습 시작...')
    try:
        # 모델이 그래디언트를 필요로 하도록 설정합니다.
        base_model.enable_input_require_grads()
        lora_model = get_peft_model(base_model, get_lora_config())
    except ValueError as e:
        print(f'  [Warning] LoRA 적용 실패: {e}')
        raise e

    lora_model.print_trainable_parameters()
    opt = AdamW(lora_model.parameters(), lr=2e-4)

    for epoch in range(LORA_EPOCHS):
        total_loss = 0.0
        for prompt, completion in expert_pairs:
            full_text = prompt + ' ' + completion
            inputs = tokenizer(
                full_text, return_tensors='pt',
                truncation=True, max_length=128
            ).to(base_model.device)
            prompt_len = tokenizer(prompt, return_tensors='pt').input_ids.shape[1]
            labels = inputs['input_ids'].clone()
            labels[:, :prompt_len] = -100
            opt.zero_grad()
            out = lora_model(**inputs, labels=labels)
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(lora_model.parameters(), 1.0)
            opt.step()
            total_loss += out.loss.item()
        print(f'    epoch {epoch+1}/{LORA_EPOCHS} | loss={total_loss/len(expert_pairs):.4f}')

    lora_model.save_pretrained(save_path)
    print(f'  [{expert_key}] 저장 완료: {save_path}')
    del lora_model
    torch.cuda.empty_cache()
    return save_path

print('12-Expert LoRA 학습 시작...')
expert_paths = {}
for ekey, pairs in EXPERT_DATA.items():
    expert_paths[ekey] = train_lora_expert(ekey, pairs, RUN_DIR)

print(f'\n✅ 12-Expert 학습 완료')
for k, v in expert_paths.items():
    print(f'  {k}: {v}')


12-Expert LoRA 학습 시작...
  [math_0] LoRA 학습 시작...
trainable params: 3,867,648 || all params: 25,809,801,520 || trainable%: 0.0150
    epoch 1/3 | loss=0.2493
    epoch 2/3 | loss=0.0573
    epoch 3/3 | loss=0.0029
  [math_0] 저장 완료: /content/drive/MyDrive/nomadic_llm_results/gemma4_26b_20260419_021120/expert_math_0
  [math_1] LoRA 학습 시작...
trainable params: 3,867,648 || all params: 25,809,801,520 || trainable%: 0.0150
    epoch 1/3 | loss=0.2080
    epoch 2/3 | loss=0.0394
    epoch 3/3 | loss=0.0009
  [math_1] 저장 완료: /content/drive/MyDrive/nomadic_llm_results/gemma4_26b_20260419_021120/expert_math_1
  [math_2] LoRA 학습 시작...
trainable params: 3,867,648 || all params: 25,809,801,520 || trainable%: 0.0150
    epoch 1/3 | loss=0.0130
    epoch 2/3 | loss=0.2579
    epoch 3/3 | loss=0.0021
  [math_2] 저장 완료: /content/drive/MyDrive/nomadic_llm_results/gemma4_26b_20260419_021120/expert_math_2
  [math_3] LoRA 학습 시작...
trainable params: 3,867,648 || all params: 25,809,801,520 || trainable%: 0.015

In [ ]:
# ============================================================
# STEP 6: PolicyNet 학습 데이터 수집 (12-Expert)
# ============================================================
SWITCH_THRESHOLD = 0.25

def collect_training_data(steps_per_prompt=15):
    data = []
    tracker = HybridDeltaTracker()

    for domain, (train_p, _, sw_label, mode_label) in DOMAIN_CONFIG.items():
        # 도메인 내 sub-expert들 중 랜덤하게 target 지정
        sub_experts = DOMAIN_TO_EXPERTS[domain]
        print(f'  [{domain}] 데이터 수집 중... (sub-experts: {sub_experts})')

        for prompt in train_p:
            tracker.reset()
            input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(base_model.device)

            for step in range(steps_per_prompt):
                with torch.no_grad():
                    out = base_model(input_ids, output_hidden_states=True)
                logits = out.logits[:, -1, :]
                hidden = out.hidden_states[-1][:, -1, :]
                err    = 1.0 - F.softmax(logits, dim=-1).max().item()
                rec    = tracker.compute(hidden, err)
                meta   = build_meta_signals(rec, base_model.device)

                # 12-Expert target label: 도메인 내 sub-expert 인덱스 (round-robin)
                expert_idx_in_domain = step % len(sub_experts)
                target_label = EXPERT_KEYS.index(sub_experts[expert_idx_in_domain])

                data.append({
                    'hidden':       hidden.detach().cpu(),
                    'meta':         meta.detach().cpu(),
                    'switch_label': sw_label,
                    'mode_label':   mode_label,
                    'target_label': target_label,
                    'delta_hybrid': rec['delta_hybrid'],
                })

                next_tok = F.softmax(logits, dim=-1).argmax(-1, keepdim=True)
                input_ids = torch.cat([input_ids, next_tok], dim=-1)
                if next_tok.item() == tokenizer.eos_token_id:
                    break

    switch_count = sum(d['switch_label'] for d in data)
    print(f'\n✅ 총 {len(data)}개 샘플 | switch=1: {switch_count} | stay=0: {len(data)-switch_count}')
    return data

train_data = collect_training_data(steps_per_prompt=15)


  [math] 데이터 수집 중... (sub-experts: ['math_0', 'math_1', 'math_2', 'math_3'])
  [code] 데이터 수집 중... (sub-experts: ['code_0', 'code_1', 'code_2', 'code_3'])
  [language] 데이터 수집 중... (sub-experts: ['language_0', 'language_1', 'language_2', 'language_3'])

✅ 총 360개 샘플 | switch=1: 120 | stay=0: 240


In [ ]:
# ============================================================
# STEP 7: PolicyNet 학습 (stay/switch + target(12) + mode)
# ============================================================
POLICY_EPOCHS = 30
POLICY_BATCH  = 16

policy_net = NomadicPolicyNet(HIDDEN_DIM, policy_hidden=128, num_experts=NUM_EXPERTS)
policy_net = policy_net.to(base_model.device)
opt_p = AdamW(policy_net.parameters(), lr=1e-3, weight_decay=1e-4)
loss_history = []

print(f'PolicyNet 파라미터: {sum(p.numel() for p in policy_net.parameters()):,}')

for epoch in range(POLICY_EPOCHS):
    indices = torch.randperm(len(train_data))
    ep_loss = 0.0; n_b = 0
    for start in range(0, len(train_data), POLICY_BATCH):
        batch = [train_data[i] for i in indices[start:start+POLICY_BATCH]]
        h_b  = torch.cat([d['hidden'] for d in batch]).to(base_model.device)
        m_b  = torch.cat([d['meta']   for d in batch]).to(base_model.device)
        sw_t = torch.tensor([d['switch_label'] for d in batch],
                             dtype=torch.long, device=base_model.device)
        md_t = torch.tensor([d['mode_label']   for d in batch],
                             dtype=torch.long, device=base_model.device)
        tg_t = torch.tensor([d['target_label'] for d in batch],
                             dtype=torch.long, device=base_model.device)

        opt_p.zero_grad()
        ss, tgt, mode = policy_net(h_b, m_b)
        loss = (F.nll_loss(torch.log(ss   + 1e-8), sw_t)
              + 0.5 * F.nll_loss(torch.log(tgt  + 1e-8), tg_t)
              + 0.3 * F.nll_loss(torch.log(mode + 1e-8), md_t))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy_net.parameters(), 1.0)
        opt_p.step()
        ep_loss += loss.item(); n_b += 1

    avg = ep_loss / n_b
    loss_history.append(avg)
    if (epoch+1) % 5 == 0:
        with torch.no_grad():
            all_h = torch.cat([d['hidden'] for d in train_data]).to(base_model.device)
            all_m = torch.cat([d['meta']   for d in train_data]).to(base_model.device)
            all_s = torch.tensor([d['switch_label'] for d in train_data],
                                  device=base_model.device)
            ss_all, _, _ = policy_net(all_h, all_m)
            acc = (ss_all.argmax(-1) == all_s).float().mean().item()
        print(f'Epoch {epoch+1:02d}/{POLICY_EPOCHS} | loss={avg:.4f} | switch_acc={acc:.3f}')

torch.save(policy_net.state_dict(), os.path.join(RUN_DIR, 'policy_net.pt'))

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(loss_history, 'b-o', ms=3)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('PolicyNet Training Loss'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, 'policy_loss.png'), dpi=150)
plt.close()
print('✅ PolicyNet 학습 완료')


PolicyNet 파라미터: 396,304
Epoch 05/30 | loss=0.6162 | switch_acc=1.000
Epoch 10/30 | loss=0.2693 | switch_acc=1.000
Epoch 15/30 | loss=0.1864 | switch_acc=1.000
Epoch 20/30 | loss=0.2014 | switch_acc=1.000
Epoch 25/30 | loss=0.0669 | switch_acc=1.000
Epoch 30/30 | loss=0.0285 | switch_acc=1.000
✅ PolicyNet 학습 완료


In [ ]:
# ============================================================
# STEP 8: 엔트로피 측정 — 학습 전후 비교
# (§4.5 Table 4 재현: untrained → trained PolicyNet)
# ============================================================
MAX_STEPS_ENTROPY = 20

def measure_entropy(prompts, label, steps=MAX_STEPS_ENTROPY):
    """프롬프트 집합의 평균 Stable/Trans H 측정"""
    # label: 'stable'이면 stable_H 집계, 'transition'이면 trans_H 집계
    entropies = []
    tracker = HybridDeltaTracker()
    for prompt in prompts:
        tracker.reset()
        input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(base_model.device)
        for _ in range(steps):
            with torch.no_grad():
                out = base_model(input_ids, output_hidden_states=True)
            logits = out.logits[:, -1, :]
            H = topk_entropy(logits / 1.0)  # raw entropy
            entropies.append(H)
            next_tok = F.softmax(logits, dim=-1).argmax(-1, keepdim=True)
            input_ids = torch.cat([input_ids, next_tok], dim=-1)
            if next_tok.item() == tokenizer.eos_token_id: break
    return float(np.mean(entropies))


# Math+Code = stable domain, Language = transition domain
stable_prompts_eval = MATH_PROMPTS_EVAL + CODE_PROMPTS_EVAL
trans_prompts_eval  = LANGUAGE_PROMPTS_EVAL

print('엔트로피 측정 중...')
stable_H_raw = measure_entropy(stable_prompts_eval, 'stable')
trans_H_raw  = measure_entropy(trans_prompts_eval,  'transition')
dH_raw = trans_H_raw - stable_H_raw

print(f'\n[학습 전 — raw signal]')
print(f'  Stable H : {stable_H_raw:.4f}')
print(f'  Trans  H : {trans_H_raw:.4f}')
print(f'  ΔH       : {dH_raw:.4f}')

# 학습 후 PolicyNet switch_prob 관측
def measure_entropy_with_policy(prompts, steps=MAX_STEPS_ENTROPY):
    entropies, sw_probs = [], []
    tracker = HybridDeltaTracker()
    for prompt in prompts:
        tracker.reset()
        input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(base_model.device)
        for _ in range(steps):
            with torch.no_grad():
                out = base_model(input_ids, output_hidden_states=True)
            logits = out.logits[:, -1, :]
            hidden = out.hidden_states[-1][:, -1, :]
            err = 1.0 - F.softmax(logits, dim=-1).max().item()
            rec  = tracker.compute(hidden, err)
            meta = build_meta_signals(rec, base_model.device)
            with torch.no_grad():
                ss, _, _ = policy_net(hidden.unsqueeze(0), meta)
            sw_probs.append(ss[0, 1].item())
            H = topk_entropy(logits)
            entropies.append(H)
            next_tok = F.softmax(logits, dim=-1).argmax(-1, keepdim=True)
            input_ids = torch.cat([input_ids, next_tok], dim=-1)
            if next_tok.item() == tokenizer.eos_token_id: break
    return float(np.mean(entropies)), float(np.mean(sw_probs))

stable_H_policy, stable_sw = measure_entropy_with_policy(stable_prompts_eval)
trans_H_policy,  trans_sw  = measure_entropy_with_policy(trans_prompts_eval)
dH_policy = trans_H_policy - stable_H_policy

print(f'\n[학습 후 — PolicyNet 포함]')
print(f'  Stable H : {stable_H_policy:.4f} (switch_prob={stable_sw:.3f})')
print(f'  Trans  H : {trans_H_policy:.4f}  (switch_prob={trans_sw:.3f})')
print(f'  ΔH       : {dH_policy:.4f}')

print(f'\n--- §4.5 Table 4 비교 ---')
print(f'{'Setting':35s} | Stable H | Trans H | ΔH')
print('-' * 65)
print(f'{'Synthetic MoE (Nomadic Full)':35s} | 0.108    | 0.896   | +0.788')
print(f'{'E2B — untrained PolicyNet':35s} | 1.806    | 2.537   | +0.731')
print(f'{'E2B — trained PolicyNet':35s} | 1.249    | 2.234   | +0.984')
print(f'{MODEL_KEY+" — untrained":35s} | {stable_H_raw:.3f}    | {trans_H_raw:.3f}   | {dH_raw:+.3f}')
print(f'{MODEL_KEY+" — trained":35s} | {stable_H_policy:.3f}    | {trans_H_policy:.3f}   | {dH_policy:+.3f}')

엔트로피 측정 중...

[학습 전 — raw signal]
  Stable H : 0.5362
  Trans  H : 0.8897
  ΔH       : 0.3535

[학습 후 — PolicyNet 포함]
  Stable H : 0.5073 (switch_prob=0.002)
  Trans  H : 0.8855  (switch_prob=0.998)
  ΔH       : 0.3782

--- §4.5 Table 4 비교 ---
Setting                             | Stable H | Trans H | ΔH
-----------------------------------------------------------------
Synthetic MoE (Nomadic Full)        | 0.108    | 0.896   | +0.788
E2B — untrained PolicyNet           | 1.806    | 2.537   | +0.731
E2B — trained PolicyNet             | 1.249    | 2.234   | +0.984
gemma4_26b — untrained              | 0.536    | 0.890   | +0.353
gemma4_26b — trained                | 0.507    | 0.886   | +0.378


In [ ]:
# ============================================================
# STEP 8.5: 스트레스 시나리오 데이터 생성
#
# 작전 1: 도메인 급변 (Sudden Context Shift)
# 작전 2: 노이즈 면역력 (Adversarial Robustness)
# 작전 3: 마이크로 지속학습 (Micro-Continual Learning)
# ============================================================
import string, random

# ────────────────────────────────────────────────────────────
# 작전 1: 도메인 급변
# ────────────────────────────────────────────────────────────
def create_context_shift_data():
    """
    평온한 일상 텍스트(Domain A) → 전문 수학/물리 텍스트(Domain B) 급전환.
    반환: (combined_text, shift_point_token_idx)
    shift_point에서 Δx_hybrid 급등 여부를 추적한다.
    """
    domain_a = (
        "The gentle breeze rustled through the green leaves of the ancient oak tree. "
        "A small family was having a peaceful picnic on the soft grass, enjoying the warm afternoon sun. "
        "The children laughed as they chased a colorful butterfly across the meadow, completely carefree. "
    ) * 3

    domain_b = (
        "Let H be a Hilbert space and T be a bounded linear operator on H. "
        "According to the spectral theorem, if T is compact and self-adjoint, "
        "there exists an orthonormal basis of H consisting of eigenvectors of T. "
        "Furthermore, the eigenvalues lambda_i must converge to zero as i approaches infinity."
    )

    combined = domain_a + "\n\n" + domain_b
    shift_point = len(tokenizer.encode(domain_a))
    return combined, shift_point

SCENARIO_SHIFT_TEXT, SCENARIO_SHIFT_POINT = create_context_shift_data()
print(f'[작전1] 도메인 급변 텍스트 생성: shift_point={SCENARIO_SHIFT_POINT} 토큰')

# ────────────────────────────────────────────────────────────
# 작전 2: 노이즈 면역력
# ────────────────────────────────────────────────────────────
def inject_adversarial_noise(text, noise_ratio=0.15, seed=42):
    """
    word-level 노이즈 주입:
      - typo : 글자 순서 바꾸기 (routing → ruoting)
      - gibberish : 완전 무작위 문자열 대체
    """
    random.seed(seed)
    words = text.split()
    noisy = []
    for w in words:
        if random.random() < noise_ratio:
            ntype = random.choice(['typo', 'gibberish'])
            if ntype == 'typo' and len(w) > 2:
                idx = random.randint(1, len(w) - 2)
                w = w[:idx] + w[idx+1] + w[idx] + w[idx+2:]
            else:
                w = ''.join(random.choices(string.ascii_lowercase, k=5))
        noisy.append(w)
    return ' '.join(noisy)

# 기준 텍스트: 안정적인 수학 프롬프트
_base_text = ' '.join([p for p, _ in EXPERT_DATA['math_0']])
SCENARIO_CLEAN_TEXT = _base_text
SCENARIO_NOISY_TEXT = inject_adversarial_noise(_base_text, noise_ratio=0.15)
print(f'[작전2] 노이즈 면역 텍스트 생성: clean={len(_base_text.split())}단어 → noisy 15% 훼손')

# ────────────────────────────────────────────────────────────
# 작전 3: 마이크로 지속학습 — 데이터 준비
# ────────────────────────────────────────────────────────────
# Old Knowledge: math_0 expert 데이터 (기존 지식)
OLD_TASK_DATA = EXPERT_DATA['math_0']   # (prompt, completion) × 6

# New Knowledge: language_1 expert 데이터 (새로운 도메인 = 철학)
NEW_TASK_DATA = EXPERT_DATA['language_1']  # (prompt, completion) × 6

def get_continual_learning_tokens():
    """old/new task를 토큰화해서 반환"""
    def tok(pairs):
        texts = [p + ' ' + c for p, c in pairs]
        return tokenizer(texts, padding=True, truncation=True,
                         max_length=128, return_tensors='pt')
    return tok(OLD_TASK_DATA), tok(NEW_TASK_DATA)

CL_OLD_TOKENS, CL_NEW_TOKENS = get_continual_learning_tokens()
print(f'[작전3] 지속학습 데이터: old={len(OLD_TASK_DATA)}쌍, new={len(NEW_TASK_DATA)}쌍')
print('✅ 스트레스 시나리오 데이터 준비 완료')


# ────────────────────────────────────────────────────────────
# 작전 4: 중첩 경계 — 복합 도메인 프롬프트 (Ambiguous/Hybrid)
# ────────────────────────────────────────────────────────────
# math + code + language가 동시에 활성화되는 프롬프트.
# 각 항목은 두 도메인 이상의 경계에 걸쳐 있음.
HYBRID_PROMPTS = [
    # math × code
    "Solve the Fibonacci sequence using dynamic programming. "
    "For each step n, the recurrence f(n) = f(n-1) + f(n-2) gives",
    # math × language
    "The derivative of beauty, mathematically speaking, "
    "approaches infinity as complexity converges to",
    # code × language
    "Write a Python function that captures the essence of melancholy: "
    "def melancholy(memories): return",
    # math × code × language
    "If consciousness were a recursive algorithm, "
    "def think(depth=0): return think(depth+1) until the eigenvalue of self-awareness",
    # 한국어 × 수학 × 시
    "소수(素數)란 무엇인가. 그것은 고독처럼 오직 자기 자신으로만 나누어지는 수. "
    "다음 소수는",
]

# 기준: 각 프롬프트에서 어느 expert로 수렴하는지(평형점) or 진동하는지(oscillation)
print(f'[작전4] 중첩 경계 프롬프트 {len(HYBRID_PROMPTS)}개 준비')

# ────────────────────────────────────────────────────────────
# 작전 5: '유혹'과 '회복' — Adversarial Lure & Recovery
# ────────────────────────────────────────────────────────────
# 수학 문제 중간에 일상 농담을 주입해 expert를 이탈시킨 뒤
# 다시 수학으로 복귀하는 속도(recovery_steps)를 측정.
#
# 구조: [math_prefix] + [lure_sentence] + [math_suffix]
# lure_point: math_prefix의 토큰 수
# return_point: math_prefix + lure_sentence의 토큰 수

LURE_CASES = [
    {
        'math_prefix': "Let us compute the integral of e^x from 0 to 1. "
                       "The antiderivative of e^x is e^x itself, therefore the result is",
        'lure':        " — oh wait, did you hear the one about the mathematician who "
                       "walked into a bar and ordered 1/3 of a beer? The bartender said 'I don't "
                       "know what that is' so the mathematician said",
        'math_suffix': " Returning to our calculation: e^1 - e^0 =",
    },
    {
        'math_prefix': "피타고라스 정리에 의하면 직각삼각형의 빗변의 제곱은",
        'lure':        " 그런데 갑자기 생각났는데, 어제 치킨을 시켰더니 배달이 너무 늦어서"
                       " 완전히 식어버렸어. 피자도 아니고 치킨이 식으면 정말 슬프잖아.",
        'math_suffix': " 다시 수학으로 돌아가서: 나머지 두 변의 제곱의 합과",
    },
    {
        'math_prefix': "The eigenvalues of a symmetric matrix are always real. "
                       "Proof: Let λ be an eigenvalue and v be the corresponding eigenvector, then",
        'lure':        " Anyway, completely unrelated, my cat knocked over my coffee this morning "
                       "and I had to clean up the entire desk. Cats are such chaotic creatures.",
        'math_suffix': " Back to the proof: we take the conjugate transpose and observe that",
    },
]

def build_lure_texts():
    """
    각 lure case에서 결합 텍스트와 포인트 인덱스를 반환.
    lure_point  : 유혹 문장이 시작되는 토큰 인덱스
    return_point: 수학으로 복귀하는 토큰 인덱스
    """
    cases = []
    for lc in LURE_CASES:
        full = lc['math_prefix'] + lc['lure'] + lc['math_suffix']
        lure_point   = len(tokenizer.encode(lc['math_prefix']))
        return_point = len(tokenizer.encode(lc['math_prefix'] + lc['lure']))
        cases.append({
            'text':         full,
            'lure_point':   lure_point,
            'return_point': return_point,
            'math_prefix':  lc['math_prefix'],
        })
    return cases

LURE_CASES_TOKENIZED = build_lure_texts()
print(f'[작전5] 유혹-회복 케이스 {len(LURE_CASES_TOKENIZED)}개 준비')
for i, lc in enumerate(LURE_CASES_TOKENIZED):
    print(f'  case{i}: lure_point={lc["lure_point"]} return_point={lc["return_point"]}')

# ────────────────────────────────────────────────────────────
# 작전 6: 정보 획득 효율성 — Information Gain per Token
# ────────────────────────────────────────────────────────────
# 고밀도(압축 논문 스타일) vs 저밀도(반복 미사여구) 텍스트 쌍.
# 각 토큰의 NLL 감소분(정보획득량)과 Δx_hybrid 누적합의 상관관계 분석.

INFO_DENSE_TEXTS = [
    # 고밀도: 압축된 기술/과학 문장
    "Transformer attention complexity O(n²d) bottlenecks long-sequence tasks; "
    "sparse attention variants reduce this to O(n√n) while preserving expressivity "
    "via locality-sensitive hashing or learned sparsity patterns.",

    "RLHF fine-tunes LLMs via PPO on human preference data: reward model r(x,y) "
    "trained on ranked pairs, policy π_θ updated to maximize E[r] - β·KL(π_θ‖π_ref).",

    "양자 얽힘(entanglement)은 비국소 상관관계를 형성하여 Bell 부등식을 위반한다. "
    "EPR 역설은 은닉변수 이론으로 해소되지 않으며, 측정 시 파동함수 붕괴가 발생한다.",
]

INFO_SPARSE_TEXTS = [
    # 저밀도: 반복적 미사여구
    "It is really very important to note that this is a very significant and "
    "very meaningful observation that many people find to be very interesting "
    "and very noteworthy in many very important ways.",

    "The thing about this particular thing is that this thing, as a thing, "
    "is quite a thing indeed, and the thing about things is that things "
    "tend to be thing-like in their thing-ness.",

    "정말로 매우 굉장히 대단히 엄청나게 놀랍고도 신기하며 흥미롭고 "
    "또한 매우 중요하고 굉장히 의미 있는 것이라고 할 수 있겠습니다. "
    "그것은 정말로 매우 대단히 중요한 것입니다.",
]

print(f'[작전6] 정보밀도 텍스트: dense={len(INFO_DENSE_TEXTS)}개, sparse={len(INFO_SPARSE_TEXTS)}개')

# ────────────────────────────────────────────────────────────
# 작전 7: 장기 문맥 고착도 — Long-Context Fixation
# ────────────────────────────────────────────────────────────
# 단일 도메인 텍스트를 반복 연결하여 긴 시퀀스를 만들고
# Δx_hybrid가 시간 축으로 0에 수렴하는지(최저 에너지 상태) 관찰.

def build_fixation_text(domain='math', repeat=8, target_tokens=600):
    """
    단일 도메인 텍스트를 반복해 긴 시퀀스 생성.
    domain: 'math' | 'code' | 'language'
    반환: (text, domain)
    """
    base_chunks = {
        'math': (
            "The fundamental theorem of calculus establishes the relationship between "
            "differentiation and integration. If f is continuous on [a,b] and F is any "
            "antiderivative of f, then the definite integral of f from a to b equals F(b)-F(a). "
        ),
        'code': (
            "A binary search algorithm works by repeatedly dividing a sorted array in half. "
            "The time complexity is O(log n) because each comparison eliminates half of the "
            "remaining elements. Implementation requires low, high, and mid pointer management. "
        ),
        'language': (
            "The nature of memory is inherently reconstructive rather than reproductive. "
            "Each act of remembering modifies the memory trace itself, blending past experience "
            "with present context. What we recall is never quite what actually happened. "
        ),
    }
    chunk = base_chunks[domain]
    text  = chunk * repeat
    # 목표 토큰 수로 잘라내기
    toks  = tokenizer.encode(text)
    if len(toks) > target_tokens:
        toks = toks[:target_tokens]
        text = tokenizer.decode(toks, skip_special_tokens=True)
    return text, len(toks)

FIXATION_CASES = []
for dom in ['math', 'code', 'language']:
    txt, n_tok = build_fixation_text(dom, repeat=8, target_tokens=600)
    FIXATION_CASES.append({'domain': dom, 'text': txt, 'n_tokens': n_tok})
    print(f'[작전7] fixation {dom}: {n_tok} 토큰')

print('✅ 작전 4~7 데이터 준비 완료')


[작전1] 도메인 급변 텍스트 생성: shift_point=152 토큰
[작전2] 노이즈 면역 텍스트 생성: clean=45단어 → noisy 15% 훼손
[작전3] 지속학습 데이터: old=6쌍, new=6쌍
✅ 스트레스 시나리오 데이터 준비 완료
[작전4] 중첩 경계 프롬프트 5개 준비
[작전5] 유혹-회복 케이스 3개 준비
  case0: lure_point=34 return_point=78
  case1: lure_point=19 return_point=62
  case2: lure_point=26 return_point=55
[작전6] 정보밀도 텍스트: dense=3개, sparse=3개
[작전7] fixation math: 410 토큰
[작전7] fixation code: 354 토큰
[작전7] fixation language: 314 토큰
✅ 작전 4~7 데이터 준비 완료


In [ ]:
# ============================================================
# STEP 9: 생성 함수 정의
#
# 1. Nomadic_Full       — 12-Expert PolicyNet 기반 동적 라우팅
# 2. MoD                — Mixture-of-Depths (토큰당 레이어 선택)
# 3. ECR                — Expert Choice Routing (expert가 top-k 토큰 선택)
# 4. Sigma_MoE          — σ-MoE (sigmoid gating, 희소 라우팅)
# 5. BiLevel_MoE        — Bi-Level Routing MoE (그룹→expert 2단계)
# 6. RoutingFree_MoE    — Routing-Free MoE (hash 기반 고정 할당)
# ============================================================
import time as _time

MAX_NEW_TOKENS  = 40
VANILLA_TEMP    = 0.7
T_STABLE        = 0.35
T_TRANSITION    = 0.90
TOP_K_ENTROPY   = 50

DOMAINS = ['math', 'code', 'language']

# ─── 내부 헬퍼 ───────────────────────────────────────────────

def _load_expert(key):
    m = PeftModel.from_pretrained(base_model, expert_paths[key])
    m.eval()
    return m

def _unload_expert(m):
    del m
    torch.cuda.empty_cache()

def _next_tok_sample(logits, temp):
    probs = F.softmax(logits / max(temp, 1e-4), dim=-1)
    tok   = torch.multinomial(probs, 1)
    lp    = F.log_softmax(logits / max(temp, 1e-4), dim=-1).gather(1, tok).item()
    return tok, lp

def _base_result(text, entropies, log_probs, expert_trace=None):
    """공통 결과 딕셔너리 구성"""
    ppl = float(np.exp(-np.mean(log_probs))) if log_probs else float('nan')
    load_st = expert_load_stats(
        {k: expert_trace.count(k) for k in set(expert_trace)}
    ) if expert_trace else {}
    trans_st = transition_sharpness(expert_trace) if expert_trace else {}
    switches = sum(1 for i in range(1, len(expert_trace or []))
                   if (expert_trace or [])[i] != (expert_trace or [])[i-1])
    return {
        'text':         text,
        'mean_entropy': float(np.mean(entropies)) if entropies else float('nan'),
        'perplexity':   ppl,
        'switch_rate':  switches / max(1, len(expert_trace or [1])),
        # Expert Load & Diversity
        'load_cv':      load_st.get('load_cv', 0.0),
        'load_entropy': load_st.get('load_entropy', 0.0),
        'gini':         load_st.get('gini', 0.0),
        # Transition Sharpness
        'settling_time':  trans_st.get('settling_time', 0.0),
        'oscillation':    trans_st.get('oscillation', 0.0),
    }


# ─── 1. Nomadic Full ────────────────────────────────────────

def generate_nomadic_full(prompt):
    tracker = HybridDeltaTracker()
    input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(base_model.device)
    entropies, log_probs, expert_trace = [], [], []
    current_key = 'math_0'
    current_model = _load_expert(current_key)

    for step in range(MAX_NEW_TOKENS):
        with torch.no_grad():
            out = current_model(input_ids, output_hidden_states=True)
        logits = out.logits[:, -1, :]
        hidden = out.hidden_states[-1][:, -1, :]
        err    = 1.0 - F.softmax(logits, dim=-1).max().item()
        rec    = tracker.compute(hidden, err)
        meta   = build_meta_signals(rec, base_model.device)

        with torch.no_grad():
            ss, tgt, _ = policy_net(hidden.unsqueeze(0), meta)
        switch_prob = ss[0, 1].item()
        tgt_key     = EXPERT_KEYS[tgt.argmax(-1).item()]

        d = rec['delta_hybrid']
        if switch_prob >= 0.5 or d >= 0.45:
            next_key = tgt_key
        elif d < 0.2 and switch_prob < 0.3:
            next_key = 'math_0'  # fallback
        else:
            next_key = current_key

        if next_key != current_key:
            _unload_expert(current_model)
            current_model = _load_expert(next_key)
            current_key   = next_key

        temp = max(T_STABLE + (T_TRANSITION - T_STABLE) * d, 1e-4)
        entropies.append(topk_entropy(logits / temp, TOP_K_ENTROPY))
        tok, lp = _next_tok_sample(logits, temp)
        log_probs.append(lp)
        expert_trace.append(current_key)
        input_ids = torch.cat([input_ids, tok], dim=-1)
        if tok.item() == tokenizer.eos_token_id: break

    _unload_expert(current_model)
    text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    return _base_result(text, entropies, log_probs, expert_trace)


# ─── 2. Mixture-of-Depths (MoD) ─────────────────────────────
# 원리: 각 토큰마다 "처리 깊이"를 Δx_hybrid로 결정.
# Δx 낮으면 얕은(빠른) 경로, Δx 높으면 깊은(정교한) 경로.
# 여기서는 expert 4개를 shallow/mid/deep 계층으로 매핑.

def generate_mod(prompt):
    """
    Mixture-of-Depths 시뮬레이션.
    Δx < 0.2  → shallow expert (math_0 / code_0 / language_0)
    Δx < 0.45 → mid expert    (math_1 / code_1 / language_1)
    Δx ≥ 0.45 → deep expert   (math_2 / code_2 / language_2)
    도메인은 이전 expert의 도메인 그룹으로 유지.
    """
    tracker = HybridDeltaTracker()
    input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(base_model.device)
    entropies, log_probs, expert_trace = [], [], []
    current_domain = 'math'
    current_depth  = 0  # 0=shallow, 1=mid, 2=deep

    depth_suffix = {0: '_0', 1: '_1', 2: '_2'}
    current_key   = current_domain + depth_suffix[current_depth]
    current_model = _load_expert(current_key)

    for step in range(MAX_NEW_TOKENS):
        with torch.no_grad():
            out = current_model(input_ids, output_hidden_states=True)
        logits = out.logits[:, -1, :]
        hidden = out.hidden_states[-1][:, -1, :]
        err = 1.0 - F.softmax(logits, dim=-1).max().item()
        rec = tracker.compute(hidden, err)
        d   = rec['delta_hybrid']

        # 깊이 결정
        new_depth = 0 if d < 0.2 else (1 if d < 0.45 else 2)
        new_key   = current_domain + depth_suffix[new_depth]

        if new_key != current_key:
            _unload_expert(current_model)
            current_model = _load_expert(new_key)
            current_key   = new_key
            current_depth = new_depth

        temp = max(T_STABLE + (T_TRANSITION - T_STABLE) * d, 1e-4)
        entropies.append(topk_entropy(logits / temp, TOP_K_ENTROPY))
        tok, lp = _next_tok_sample(logits, temp)
        log_probs.append(lp)
        expert_trace.append(current_key)
        input_ids = torch.cat([input_ids, tok], dim=-1)
        if tok.item() == tokenizer.eos_token_id: break

    _unload_expert(current_model)
    text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    return _base_result(text, entropies, log_probs, expert_trace)


# ─── 3. Expert Choice Routing (ECR) ─────────────────────────
# 원리: expert가 top-k 토큰을 선택 (토큰이 expert를 선택하는 게 아님).
# 시뮬레이션: hidden을 모든 expert gate에 통과시켜,
# 각 expert마다 "자신이 처리하고 싶은 정도" score를 계산,
# 가장 높은 score의 expert를 선택.

class ECRGatingNet(nn.Module):
    """12개 expert 각각의 preference score를 계산하는 경량 게이팅 네트워크."""
    def __init__(self, hidden_dim, num_experts=12):
        super().__init__()
        self.gate = nn.Linear(hidden_dim, num_experts, bias=False)
    def forward(self, h):
        return self.gate(h.float())  # (1, 12) logits

# ECR gate는 PolicyNet hidden proj 재활용 (별도 학습 없이 초기화)
ecr_gate = ECRGatingNet(HIDDEN_DIM, NUM_EXPERTS).to(base_model.device)
# PolicyNet의 proj 가중치를 ECR gate 초기값으로 사용
with torch.no_grad():
    # PolicyNet proj: (hidden_dim, policy_hidden) → ECR: (hidden_dim, 12)
    # 단순 초기화 (실제론 별도 학습 필요)
    nn.init.xavier_uniform_(ecr_gate.gate.weight)

def generate_ecr(prompt):
    """Expert Choice Routing: expert preference score 최고값 expert 선택"""
    input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(base_model.device)
    entropies, log_probs, expert_trace = [], [], []
    current_key   = 'math_0'
    current_model = _load_expert(current_key)

    for step in range(MAX_NEW_TOKENS):
        with torch.no_grad():
            out = current_model(input_ids, output_hidden_states=True)
        logits = out.logits[:, -1, :]
        hidden = out.hidden_states[-1][:, -1, :]

        with torch.no_grad():
            scores  = ecr_gate(hidden)          # (1, 12)
            best_id = scores.argmax(-1).item()
        next_key = EXPERT_KEYS[best_id]

        if next_key != current_key:
            _unload_expert(current_model)
            current_model = _load_expert(next_key)
            current_key   = next_key

        entropies.append(topk_entropy(logits, TOP_K_ENTROPY))
        tok, lp = _next_tok_sample(logits, VANILLA_TEMP)
        log_probs.append(lp)
        expert_trace.append(current_key)
        input_ids = torch.cat([input_ids, tok], dim=-1)
        if tok.item() == tokenizer.eos_token_id: break

    _unload_expert(current_model)
    text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    return _base_result(text, entropies, log_probs, expert_trace)


# ─── 4. Sigma-MoE ────────────────────────────────────────────
# 원리: softmax 대신 sigmoid gating → 복수 expert 동시 활성화 가능.
# k개 expert를 sigmoid score 기준으로 활성화하고 가중 평균 logits.

def generate_sigma_moe(prompt, top_k_experts=2):
    """
    σ-MoE: sigmoid gate로 top_k expert를 선택, logits를 가중 평균.
    희소 활성화: sigmoid score > threshold인 expert만 사용.
    """
    tracker = HybridDeltaTracker()
    input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(base_model.device)
    entropies, log_probs, expert_trace = [], [], []

    # sigmoid gate (ECR gate 재활용)
    with torch.no_grad():
        nn.init.normal_(ecr_gate.gate.weight, std=0.02)

    for step in range(MAX_NEW_TOKENS):
        with torch.no_grad():
            out = base_model(input_ids, output_hidden_states=True)
        logits_base = out.logits[:, -1, :]
        hidden      = out.hidden_states[-1][:, -1, :]
        err         = 1.0 - F.softmax(logits_base, dim=-1).max().item()
        rec         = tracker.compute(hidden, err)

        with torch.no_grad():
            sig_scores = torch.sigmoid(ecr_gate(hidden)).squeeze(0)  # (12,)
            topk_ids   = sig_scores.topk(top_k_experts).indices.tolist()

        # 선택된 expert들의 logits 가중 평균
        weights = sig_scores[topk_ids]
        weights = weights / weights.sum()

        agg_logits = torch.zeros_like(logits_base)
        chosen_keys = []
        for w, eid in zip(weights.tolist(), topk_ids):
            ekey  = EXPERT_KEYS[eid]
            emod  = _load_expert(ekey)
            with torch.no_grad():
                eo = emod(input_ids)
            agg_logits = agg_logits + w * eo.logits[:, -1, :]
            _unload_expert(emod)
            chosen_keys.append(ekey)

        d    = rec['delta_hybrid']
        temp = max(T_STABLE + (T_TRANSITION - T_STABLE) * d, 1e-4)
        entropies.append(topk_entropy(agg_logits / temp, TOP_K_ENTROPY))
        tok, lp = _next_tok_sample(agg_logits, temp)
        log_probs.append(lp)
        expert_trace.append(chosen_keys[0])  # 대표 expert 기록
        input_ids = torch.cat([input_ids, tok], dim=-1)
        if tok.item() == tokenizer.eos_token_id: break

    text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    return _base_result(text, entropies, log_probs, expert_trace)


# ─── 5. Bi-Level Routing MoE ─────────────────────────────────
# 원리: 1단계 — domain 그룹(math/code/language) 선택
#        2단계 — 선택된 도메인 내 sub-expert(_0.._3) 선택

def generate_bilevel_moe(prompt):
    """
    Bi-Level Routing:
    Level 1: Δx_hybrid로 domain 선택 (math/code/language)
    Level 2: domain 내 hidden 유사도로 sub-expert 선택
    """
    tracker = HybridDeltaTracker()
    input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(base_model.device)
    entropies, log_probs, expert_trace = [], [], []
    current_key   = 'math_0'
    current_model = _load_expert(current_key)

    for step in range(MAX_NEW_TOKENS):
        with torch.no_grad():
            out = current_model(input_ids, output_hidden_states=True)
        logits = out.logits[:, -1, :]
        hidden = out.hidden_states[-1][:, -1, :]
        err    = 1.0 - F.softmax(logits, dim=-1).max().item()
        rec    = tracker.compute(hidden, err)
        d      = rec['delta_hybrid']

        # Level 1: domain 선택
        if d < 0.2:
            domain = 'math'
        elif d < 0.5:
            domain = 'code'
        else:
            domain = 'language'

        # Level 2: 도메인 내 sub-expert 선택 — hidden norm 기반 (sub 0~3)
        # hidden L2 norm을 4구간으로 나눠 sub_id 결정
        h_norm  = hidden.float().norm().item()
        sub_id  = min(3, int(h_norm / (HIDDEN_DIM ** 0.5) * 4))
        next_key = f'{domain}_{sub_id}'

        if next_key != current_key:
            _unload_expert(current_model)
            current_model = _load_expert(next_key)
            current_key   = next_key

        temp = max(T_STABLE + (T_TRANSITION - T_STABLE) * d, 1e-4)
        entropies.append(topk_entropy(logits / temp, TOP_K_ENTROPY))
        tok, lp = _next_tok_sample(logits, temp)
        log_probs.append(lp)
        expert_trace.append(current_key)
        input_ids = torch.cat([input_ids, tok], dim=-1)
        if tok.item() == tokenizer.eos_token_id: break

    _unload_expert(current_model)
    text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    return _base_result(text, entropies, log_probs, expert_trace)


# ─── 6. Routing-Free MoE ─────────────────────────────────────
# 원리: 토큰 ID를 해시하여 고정 expert 할당 (학습 없음).
# routing overhead = 0, 단 맥락 인식 전혀 없음 → 하한선 대조군.

def generate_routing_free(prompt):
    """
    Routing-Free MoE: token_id % NUM_EXPERTS → expert 고정 할당.
    맥락 무관, zero routing overhead.
    """
    input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(base_model.device)
    entropies, log_probs, expert_trace = [], [], []
    current_key   = EXPERT_KEYS[0]
    current_model = _load_expert(current_key)

    for step in range(MAX_NEW_TOKENS):
        with torch.no_grad():
            out = current_model(input_ids)
        logits = out.logits[:, -1, :]

        # 현재 마지막 토큰 ID로 hash → expert 결정
        last_tok_id = input_ids[0, -1].item()
        next_id     = last_tok_id % NUM_EXPERTS
        next_key    = EXPERT_KEYS[next_id]

        if next_key != current_key:
            _unload_expert(current_model)
            current_model = _load_expert(next_key)
            current_key   = next_key

        entropies.append(topk_entropy(logits, TOP_K_ENTROPY))
        tok, lp = _next_tok_sample(logits, VANILLA_TEMP)
        log_probs.append(lp)
        expert_trace.append(current_key)
        input_ids = torch.cat([input_ids, tok], dim=-1)
        if tok.item() == tokenizer.eos_token_id: break

    _unload_expert(current_model)
    text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    return _base_result(text, entropies, log_probs, expert_trace)


# ─── 모델 레지스트리 ─────────────────────────────────────────
ALL_MODELS = {
    'Nomadic_Full':    generate_nomadic_full,
    'MoD':             generate_mod,
    'ECR':             generate_ecr,
    'Sigma_MoE':       generate_sigma_moe,
    'BiLevel_MoE':     generate_bilevel_moe,
    'RoutingFree_MoE': generate_routing_free,
}

print('✅ 생성 함수 정의 완료')
print(f'   대조군: {list(ALL_MODELS.keys())}')


✅ 생성 함수 정의 완료
   대조군: ['Nomadic_Full', 'MoD', 'ECR', 'Sigma_MoE', 'BiLevel_MoE', 'RoutingFree_MoE']


In [23]:
# ============================================================
# STEP 10: 벤치마크 실행
#
# 시나리오 A: 일반 도메인 평가 (math/code/language 프롬프트)
# 시나리오 B: 도메인 급변 (shift_text 전후 Δx 추적)
# 시나리오 C: 노이즈 면역 (clean vs noisy 비교)
# ============================================================
import time as _btime

NUM_RUNS = 3

def repeat_rate(text, n=2):
    tokens = text.split()
    if len(tokens) < n: return 0.0
    ngrams = [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
    return 1.0 - len(set(ngrams)) / max(1, len(ngrams))

all_results    = []  # 시나리오 A
shift_results  = []  # 시나리오 B
noise_results  = []  # 시나리오 C
latency_results = [] # 레이턴시/FLOPs

# ────────────────────────────────────────────────────────────
# 시나리오 A: 일반 도메인 평가
# ────────────────────────────────────────────────────────────
print('\n' + '='*65)
print('시나리오 A: 일반 도메인 평가')
print('='*65)
for domain in DOMAINS:
    _, eval_p, _, _ = DOMAIN_CONFIG[domain]
    print(f'\n--- domain: {domain} ({len(eval_p)} prompts × {NUM_RUNS} runs) ---')
    for prompt in eval_p:
        for run in range(NUM_RUNS):
            for model_name, gen_fn in ALL_MODELS.items():
                t0  = _btime.perf_counter()
                res = gen_fn(prompt)
                t1  = _btime.perf_counter()
                n_tok = len(tokenizer.encode(res['text']))
                all_results.append({
                    'model':        model_name,
                    'domain':       domain,
                    'run':          run,
                    'entropy':      res['mean_entropy'],
                    'perplexity':   res['perplexity'],
                    'repeat_rate':  repeat_rate(res['text']),
                    'switch_rate':  res.get('switch_rate', 0.0),
                    # 확장 지표
                    'load_cv':      res.get('load_cv', 0.0),
                    'load_entropy': res.get('load_entropy', 0.0),
                    'gini':         res.get('gini', 0.0),
                    'settling_time':res.get('settling_time', 0.0),
                    'oscillation':  res.get('oscillation', 0.0),
                    'latency_ms':   (t1-t0)/max(1,n_tok)*1000,
                    'text':         res['text'][:200],
                })
        print(f'  {prompt[:45]}... 완료')

# ────────────────────────────────────────────────────────────
# 시나리오 B: 도메인 급변 — Δx 시계열 추적
# ────────────────────────────────────────────────────────────
print('\n' + '='*65)
print('시나리오 B: 도메인 급변 추적')
print('='*65)

def run_shift_scenario(model_name, gen_fn, text, shift_point):
    """shift_point 전후 Δx_hybrid를 토큰 단위로 추적"""
    tracker = HybridDeltaTracker()
    input_ids = tokenizer(text, return_tensors='pt').input_ids.to(base_model.device)
    # shift_point 토큰까지만 순방향 실행하며 Δx 기록
    trace = []
    with torch.no_grad():
        for t in range(min(shift_point + 30, input_ids.shape[1])):
            ids_t = input_ids[:, :t+1]
            out   = base_model(ids_t, output_hidden_states=True)
            hidden = out.hidden_states[-1][:, -1, :]
            err    = 1.0 - F.softmax(out.logits[:, -1, :], dim=-1).max().item()
            rec    = tracker.compute(hidden, err)
            trace.append({
                'token_idx':    t,
                'delta_hybrid': rec['delta_hybrid'],
                'delta_env':    rec['delta_env'],
                'delta_err':    rec['delta_err'],
                'is_post_shift': t >= shift_point,
            })
    # shift 전후 평균 Δx
    pre  = [r['delta_hybrid'] for r in trace if not r['is_post_shift']]
    post = [r['delta_hybrid'] for r in trace if r['is_post_shift']]
    return {
        'model':         model_name,
        'pre_mean_dx':   float(np.mean(pre))  if pre  else 0.0,
        'post_mean_dx':  float(np.mean(post)) if post else 0.0,
        'delta_dx':      float(np.mean(post) - np.mean(pre)) if (pre and post) else 0.0,
        'trace':         trace,
    }

for mname, gfn in ALL_MODELS.items():
    sr = run_shift_scenario(mname, gfn, SCENARIO_SHIFT_TEXT, SCENARIO_SHIFT_POINT)
    shift_results.append(sr)
    print(f'  {mname:20s}: pre_Δx={sr["pre_mean_dx"]:.4f} → post_Δx={sr["post_mean_dx"]:.4f}  (Δ={sr["delta_dx"]:+.4f})')

# ────────────────────────────────────────────────────────────
# 시나리오 C: 노이즈 면역력 — clean vs noisy
# ────────────────────────────────────────────────────────────
print('\n' + '='*65)
print('시나리오 C: 노이즈 면역력')
print('='*65)

for mname, gfn in ALL_MODELS.items():
    for cond, txt in [('clean', SCENARIO_CLEAN_TEXT), ('noisy', SCENARIO_NOISY_TEXT)]:
        res = gfn(txt[:200])  # 앞 200자만 입력
        noise_results.append({
            'model':        mname,
            'condition':    cond,
            'entropy':      res['mean_entropy'],
            'perplexity':   res['perplexity'],
            'load_cv':      res.get('load_cv', 0.0),
            'settling_time':res.get('settling_time', 0.0),
        })
    r_c = next(r for r in noise_results if r['model']==mname and r['condition']=='clean')
    r_n = next(r for r in noise_results if r['model']==mname and r['condition']=='noisy')
    print(f'  {mname:20s}: entropy clean={r_c["entropy"]:.4f} / noisy={r_n["entropy"]:.4f}  '
          f'(load_cv clean={r_c["load_cv"]:.3f} / noisy={r_n["load_cv"]:.3f})')

# ────────────────────────────────────────────────────────────
# 레이턴시 측정 (각 모델 × 대표 프롬프트)
# ────────────────────────────────────────────────────────────
print('\n' + '='*65)
print('레이턴시 & FLOPs 측정')
print('='*65)
_latency_prompt = DOMAIN_CONFIG['math'][1][0]  # eval 첫 번째 math 프롬프트
n_params = sum(p.numel() for p in base_model.parameters())

for mname, gfn in ALL_MODELS.items():
    lat = measure_latency_flops(gfn, _latency_prompt, n_warmup=1, n_measure=2)
    lat['model'] = mname
    lat['n_params'] = n_params
    latency_results.append(lat)
    print(f'  {mname:20s}: {lat["latency_ms"]:7.2f} ms/tok | '
          f'{lat["tokens_per_sec"]:6.1f} tok/s | '
          f'FLOPs≈{lat["est_flops"]/1e12:.2f}T')

print('\n✅ 전체 벤치마크 완료')

# ────────────────────────────────────────────────────────────
# 시나리오 D: 중첩 경계 — Hybrid Domain
# ────────────────────────────────────────────────────────────
print('\n' + '='*65)
print('시나리오 D: 중첩 경계 (Ambiguous/Hybrid Domain)')
print('='*65)

hybrid_results = []

def run_hybrid_scenario(model_name, gen_fn, prompts):
    """
    복합 도메인 프롬프트에서 expert 분포를 추적.
    평형점(dominant expert ≥ 60%) 또는 진동(oscillation > 0.3) 판별.
    """
    rows = []
    for prompt in prompts:
        res = gen_fn(prompt)
        # expert_trace에서 도메인 수준으로 집계
        # _base_result는 expert_trace를 직접 저장하지 않으므로,
        # load_entropy와 oscillation으로 간접 측정
        rows.append({
            'model':        model_name,
            'prompt_head':  prompt[:40],
            'entropy':      res['mean_entropy'],
            'load_cv':      res.get('load_cv', 0.0),
            'load_entropy': res.get('load_entropy', 0.0),
            'oscillation':  res.get('oscillation', 0.0),
            'settling_time':res.get('settling_time', 0.0),
            # 평형점 판별: load_entropy 낮으면 한 expert 집중(평형),
            # oscillation 높으면 진동
            'state': 'equilibrium' if res.get('load_entropy', 0) < 1.5 else
                     ('oscillating' if res.get('oscillation', 0) > 0.3 else 'hybrid'),
        })
    return rows

for mname, gfn in ALL_MODELS.items():
    rows = run_hybrid_scenario(mname, gfn, HYBRID_PROMPTS)
    hybrid_results.extend(rows)
    states = [r['state'] for r in rows]
    osc_mean = np.mean([r['oscillation'] for r in rows])
    print(f'  {mname:20s}: states={states}  mean_osc={osc_mean:.3f}')

# ────────────────────────────────────────────────────────────
# 시나리오 E: 유혹과 회복 — Adversarial Lure & Recovery
# ────────────────────────────────────────────────────────────
print('\n' + '='*65)
print('시나리오 E: 유혹과 회복 (Adversarial Lure & Recovery)')
print('='*65)

lure_results = []

def run_lure_scenario(model_name, lure_case):
    """
    토큰 단위로 Δx_hybrid를 추적.
    구간별 분석:
      pre_lure    : [0, lure_point)
      lure_zone   : [lure_point, return_point)
      post_return : [return_point, end)
    recovery_steps: post_return 구간에서 Δx가 pre_lure 수준으로
                    복귀하는 데 걸리는 스텝 수 (수렴 기준: < pre_mean + 0.05)
    """
    tracker    = HybridDeltaTracker()
    text       = lure_case['text']
    lure_pt    = lure_case['lure_point']
    return_pt  = lure_case['return_point']
    input_ids  = tokenizer(text, return_tensors='pt').input_ids.to(base_model.device)
    total_len  = min(input_ids.shape[1], return_pt + 40)

    trace = []
    with torch.no_grad():
        for t in range(total_len):
            ids_t  = input_ids[:, :t+1]
            out    = base_model(ids_t, output_hidden_states=True)
            hidden = out.hidden_states[-1][:, -1, :]
            err    = 1.0 - F.softmax(out.logits[:, -1, :], dim=-1).max().item()
            rec    = tracker.compute(hidden, err)
            zone   = ('pre_lure'    if t < lure_pt   else
                      'lure_zone'   if t < return_pt  else
                      'post_return')
            trace.append({
                'token_idx':    t,
                'delta_hybrid': rec['delta_hybrid'],
                'delta_err':    rec['delta_err'],
                'zone':         zone,
            })

    pre    = [r['delta_hybrid'] for r in trace if r['zone'] == 'pre_lure']
    lure   = [r['delta_hybrid'] for r in trace if r['zone'] == 'lure_zone']
    post   = [r['delta_hybrid'] for r in trace if r['zone'] == 'post_return']

    pre_mean  = float(np.mean(pre))  if pre  else 0.0
    lure_mean = float(np.mean(lure)) if lure else 0.0
    post_mean = float(np.mean(post)) if post else 0.0

# --- 수정한 루프 부분 ---
    # recovery_steps: post_return에서 Δx가 pre_mean + 0.05 아래로 내려오는 첫 스텝
    threshold = pre_mean + 0.05
    recovery_steps = len(post)  # 복귀 못 하면 최대값
    for j, r_val in enumerate(post): # r 대신 r_val(숫자)로 명확히 표기
        if r_val < threshold: # r_val 자체가 숫자이므로 ['delta_hybrid']를 뺍니다.
            recovery_steps = j + 1
            break
# -----------------------

    lure_spike = lure_mean - pre_mean  # 유혹 구간 Δx 상승량

    return {
        'model':          model_name,
        'prompt_head':    lure_case['math_prefix'][:35],
        'pre_mean_dx':    pre_mean,
        'lure_mean_dx':   lure_mean,
        'post_mean_dx':   post_mean,
        'lure_spike':     lure_spike,
        'recovery_steps': recovery_steps,
        'trace':          trace,
    }

for mname, gfn in ALL_MODELS.items():
    case_rows = []
    for lc in LURE_CASES_TOKENIZED:
        row = run_lure_scenario(mname, lc)
        case_rows.append(row)
        lure_results.append(row)
    avg_rec = np.mean([r['recovery_steps'] for r in case_rows])
    avg_sp  = np.mean([r['lure_spike']     for r in case_rows])
    print(f'  {mname:20s}: lure_spike={avg_sp:+.4f}  recovery_steps={avg_rec:.1f}')

# ────────────────────────────────────────────────────────────
# 시나리오 F: 정보 획득 효율성 — Information Gain per Token
# ────────────────────────────────────────────────────────────
print('\n' + '='*65)
print('시나리오 F: 정보 획득 효율성 (Information Gain per Token)')
print('='*65)

infogain_results = []

def run_infogain_scenario(model_name, texts, density_label):
    """
    각 토큰의 NLL(negative log-likelihood)을 측정하고
    Δx_hybrid와의 Pearson 상관계수를 계산.
    정보 획득량 = NLL_t - NLL_{t+1} (NLL 감소분 = 다음 토큰 예측이 얼마나 쉬워졌는가)
    """
    all_dx, all_nll, all_nll_gain = [], [], []

    for text in texts:
        tracker   = HybridDeltaTracker()
        input_ids = tokenizer(text, return_tensors='pt').input_ids.to(base_model.device)
        nll_seq   = []

        with torch.no_grad():
            for t in range(min(input_ids.shape[1] - 1, 80)):
                ids_t  = input_ids[:, :t+2]
                out    = base_model(ids_t, output_hidden_states=True)
                hidden = out.hidden_states[-1][:, -1, :]
                logits = out.logits[:, -1, :]
                err    = 1.0 - F.softmax(logits, dim=-1).max().item()
                rec    = tracker.compute(hidden, err)

                # 현재 토큰의 NLL (실제 다음 토큰 기준)
                next_tok_id = input_ids[0, t+1].item() if t+1 < input_ids.shape[1] else 0
                log_probs   = F.log_softmax(logits, dim=-1)
                nll_t       = -log_probs[0, next_tok_id].item()

                all_dx.append(rec['delta_hybrid'])
                all_nll.append(nll_t)
                nll_seq.append(nll_t)

        # NLL 감소분 (정보 획득량): nll[t] - nll[t+1]
        if len(nll_seq) > 1:
            gains = [nll_seq[i] - nll_seq[i+1] for i in range(len(nll_seq)-1)]
            all_nll_gain.extend(gains)

    # Pearson 상관계수 (Δx vs NLL)
    if len(all_dx) > 2 and len(all_nll) > 2:
        corr_dx_nll = float(np.corrcoef(all_dx[:len(all_nll)], all_nll)[0, 1])
    else:
        corr_dx_nll = float('nan')

    # NLL gain 통계
    mean_gain = float(np.mean(all_nll_gain)) if all_nll_gain else float('nan')
    mean_nll  = float(np.mean(all_nll))  if all_nll  else float('nan')
    mean_dx   = float(np.mean(all_dx))   if all_dx   else float('nan')

    return {
        'model':        model_name,
        'density':      density_label,
        'mean_nll':     mean_nll,
        'mean_dx':      mean_dx,
        'mean_nll_gain':mean_gain,
        'corr_dx_nll':  corr_dx_nll,
    }

for mname, gfn in ALL_MODELS.items():
    r_dense  = run_infogain_scenario(mname, INFO_DENSE_TEXTS,  'dense')
    r_sparse = run_infogain_scenario(mname, INFO_SPARSE_TEXTS, 'sparse')
    infogain_results.extend([r_dense, r_sparse])
    print(f'  {mname:20s}: '
          f'dense  NLL={r_dense["mean_nll"]:.3f}  Δx={r_dense["mean_dx"]:.3f}  corr={r_dense["corr_dx_nll"]:+.3f}')
    print(f'  {" ":20s}  '
          f'sparse NLL={r_sparse["mean_nll"]:.3f}  Δx={r_sparse["mean_dx"]:.3f}  corr={r_sparse["corr_dx_nll"]:+.3f}')

# ────────────────────────────────────────────────────────────
# 시나리오 G: 장기 문맥 고착도 — Long-Context Fixation
# ────────────────────────────────────────────────────────────
print('\n' + '='*65)
print('시나리오 G: 장기 문맥 고착도 (Long-Context Fixation)')
print('='*65)

fixation_results = []

def run_fixation_scenario(model_name, fixation_case):
    """
    긴 단일 도메인 시퀀스를 토큰 단위로 순방향 실행하며 Δx_hybrid 추적.
    - phi_trace: 전체 Δx 시계열
    - convergence_step: Δx < 0.02 로 수렴하는 첫 스텝 (최저 에너지 상태)
    - final_phi_mean: 마지막 20% 구간의 평균 Δx (수렴값)
    - ema_decay_rate: Δx EMA의 지수 감소율 추정 (log-linear fit)
    """
    tracker   = HybridDeltaTracker()
    text      = fixation_case['text']
    input_ids = tokenizer(text, return_tensors='pt').input_ids.to(base_model.device)
    total_len = input_ids.shape[1]

    phi_trace = []
    with torch.no_grad():
        for t in range(total_len):
            ids_t  = input_ids[:, :t+1]
            out    = base_model(ids_t, output_hidden_states=True)
            hidden = out.hidden_states[-1][:, -1, :]
            err    = 1.0 - F.softmax(out.logits[:, -1, :], dim=-1).max().item()
            rec    = tracker.compute(hidden, err)
            phi_trace.append(rec['delta_hybrid'])

    # 수렴 스텝
    CONV_THRESHOLD = 0.02
    convergence_step = total_len  # 수렴 안 하면 최대값
    for j, phi in enumerate(phi_trace):
        if phi < CONV_THRESHOLD:
            # 이후 10스텝도 모두 threshold 이하인지 확인 (일시적 강하 제외)
            window = phi_trace[j:j+10]
            if all(p < CONV_THRESHOLD + 0.01 for p in window):
                convergence_step = j
                break

    # 마지막 20% 평균
    tail_start  = max(0, int(total_len * 0.8))
    final_mean  = float(np.mean(phi_trace[tail_start:])) if phi_trace[tail_start:] else float('nan')

    # EMA 감소율 추정: log(Δx + ε) ~ -λ·t
    eps = 1e-6
    log_phi = np.log(np.array(phi_trace[5:]) + eps)  # 초기 spike 제외
    t_arr   = np.arange(len(log_phi), dtype=float)
    if len(t_arr) > 5:
        coeffs      = np.polyfit(t_arr, log_phi, 1)
        decay_rate  = float(-coeffs[0])  # 양수면 수렴
    else:
        decay_rate = float('nan')

    return {
        'model':            model_name,
        'domain':           fixation_case['domain'],
        'n_tokens':         total_len,
        'convergence_step': convergence_step,
        'final_phi_mean':   final_mean,
        'decay_rate':       decay_rate,
        'phi_trace':        phi_trace,
    }

for mname, gfn in ALL_MODELS.items():
    for fc in FIXATION_CASES:
        row = run_fixation_scenario(mname, fc)
        fixation_results.append(row)
    avg_conv = np.mean([r['convergence_step'] for r in fixation_results
                        if r['model'] == mname])
    avg_fin  = np.mean([r['final_phi_mean']   for r in fixation_results
                        if r['model'] == mname])
    print(f'  {mname:20s}: convergence_step={avg_conv:.0f}  final_phi={avg_fin:.4f}')

print('\n✅ 시나리오 D~G 완료')



시나리오 A: 일반 도메인 평가

--- domain: math (5 prompts × 3 runs) ---
  24를 6으로 나누면... 완료
  The square root of 144 is... 완료
  If x + 5 = 12, then x =... 완료
  원의 넓이 공식은 πr²이다. 반지름이 3이면 넓이는... 완료
  등차수열 2, 5, 8, 11의 다음 항은... 완료

--- domain: code (5 prompts × 3 runs) ---
  # 피보나치 수열
def fib(n):
    if n <= 1: return n... 완료
  arr = [3, 1, 4, 1, 5]
arr.sort()
print(arr)  ... 완료
  const greeting = (name) => `Hello,... 완료
  try:
    result = 10 / 0
except ZeroDivisionE... 완료
  <html>
<body>
<h1>Hello</h1>
</body>
</... 완료

--- domain: language (5 prompts × 3 runs) ---
  바람이 부는 날, 그는 오래된 편지를... 완료
  The paradox of freedom is that... 완료
  만약 내가 다시 태어난다면... 완료
  In the silence between words, there is... 완료
  존재한다는 것의 의미를 묻는다면... 완료

시나리오 B: 도메인 급변 추적
  Nomadic_Full        : pre_Δx=0.2732 → post_Δx=0.3658  (Δ=+0.0927)
  MoD                 : pre_Δx=0.2732 → post_Δx=0.3658  (Δ=+0.0927)
  ECR                 : pre_Δx=0.2732 → post_Δx=0.3658  (Δ=+0.0927)
  Sigma_MoE           : pre_Δx=0.2732 → post_Δx=0.36

In [24]:
# ============================================================
# STEP 11: 결과 집계 — 모든 지표
# ============================================================
df_a = pd.DataFrame(all_results)
df_a['phase'] = df_a['domain'].map({'math':'stable','code':'stable','language':'transition'})

MODEL_ORDER = ['Nomadic_Full','MoD','ECR','Sigma_MoE','BiLevel_MoE','RoutingFree_MoE']

# ── A. 주요 지표 집계 ────────────────────────────────────────
summary = df_a.groupby(['model','phase']).agg(
    mean_H        =('entropy',      'mean'),
    mean_ppl      =('perplexity',   'mean'),
    mean_rep      =('repeat_rate',  'mean'),
    mean_sw       =('switch_rate',  'mean'),
    mean_load_cv  =('load_cv',      'mean'),
    mean_load_ent =('load_entropy', 'mean'),
    mean_gini     =('gini',         'mean'),
    mean_settling =('settling_time','mean'),
    mean_osc      =('oscillation',  'mean'),
    mean_lat_ms   =('latency_ms',   'mean'),
).round(4).reset_index()

pivot = summary.pivot_table(index='model', columns='phase', values='mean_H').reset_index()
pivot['ΔH'] = pivot.get('transition', 0) - pivot.get('stable', 0)

print('\n' + '='*80)
print(f'§4.5 LLM EXPERIMENT — {MODEL_KEY}  (12-Expert)')
print('='*80)

print('\n--- ΔH Summary ---')
for _, row in pivot.iterrows():
    m = row['model']
    sh = row.get('stable', float('nan'))
    th = row.get('transition', float('nan'))
    dh = row.get('ΔH', float('nan'))
    print(f'  {m:20s}: Stable H={sh:.4f}  Trans H={th:.4f}  ΔH={dh:+.4f}')

print('\n--- Expert Load & Diversity (stable phase) ---')
load_df = summary[summary['phase']=='stable'][['model','mean_load_cv','mean_load_ent','mean_gini']]
print(load_df.to_string(index=False))

print('\n--- Transition Sharpness ---')
sharp_df = summary[summary['phase']=='stable'][['model','mean_settling','mean_osc','mean_sw']]
print(sharp_df.to_string(index=False))

print('\n--- Inference Latency (ms/tok) ---')
lat_df = pd.DataFrame(latency_results)[['model','latency_ms','tokens_per_sec','est_flops','avg_tokens']]
print(lat_df.to_string(index=False))

print('\n--- 시나리오 B: 도메인 급변 Δx ---')
shift_df = pd.DataFrame([{k:v for k,v in s.items() if k!='trace'} for s in shift_results])
print(shift_df.to_string(index=False))

print('\n--- 시나리오 C: 노이즈 면역력 ---')
noise_df = pd.DataFrame(noise_results)
pivot_noise = noise_df.pivot_table(index='model', columns='condition',
                                    values=['entropy','load_cv']).round(4)
print(pivot_noise.to_string())

# 저장
df_a.to_csv(os.path.join(RUN_DIR, 'results_scenario_a.csv'), index=False, encoding='utf-8-sig')
shift_df.to_csv(os.path.join(RUN_DIR, 'results_scenario_b.csv'), index=False)
noise_df.to_csv(os.path.join(RUN_DIR, 'results_scenario_c.csv'), index=False)
lat_df.to_csv(os.path.join(RUN_DIR, 'results_latency.csv'), index=False)

# PAPER.md용 테이블
print('\n--- PAPER.md 반영용 ---')
print('| Model | Stable H | Trans H | ΔH | Load CV | Settling | Lat(ms/tok) |')
print('|---|---|---|---|---|---|---|')
for m in MODEL_ORDER:
    r_s = summary[(summary['model']==m) & (summary['phase']=='stable')]
    r_t = summary[(summary['model']==m) & (summary['phase']=='transition')]
    sh  = r_s['mean_H'].values[0]        if len(r_s) else float('nan')
    th  = r_t['mean_H'].values[0]        if len(r_t) else float('nan')
    dh  = th - sh
    cv  = r_s['mean_load_cv'].values[0]  if len(r_s) else float('nan')
    st  = r_s['mean_settling'].values[0] if len(r_s) else float('nan')
    lt  = next((l['latency_ms'] for l in latency_results if l['model']==m), float('nan'))
    print(f'| {m:20s} | {sh:.3f} | {th:.3f} | {dh:+.3f} | {cv:.3f} | {st:.2f} | {lt:.2f} |')

print('\n✅ 집계 완료')

# ── D. 중첩 경계 집계 ────────────────────────────────────────
print('\n--- 시나리오 D: 중첩 경계 상태 분포 ---')
hybrid_df = pd.DataFrame(hybrid_results)
state_pivot = hybrid_df.groupby(['model','state']).size().unstack(fill_value=0)
print(state_pivot.to_string())
hybrid_osc = hybrid_df.groupby('model')['oscillation'].mean().reset_index()
print('\nmean oscillation per model:')
print(hybrid_osc.to_string(index=False))

# ── E. 유혹-회복 집계 ────────────────────────────────────────
print('\n--- 시나리오 E: 유혹-회복 ---')
lure_df = pd.DataFrame([{k:v for k,v in r.items() if k!='trace'} for r in lure_results])
lure_summary = lure_df.groupby('model').agg(
    mean_lure_spike     =('lure_spike',      'mean'),
    mean_recovery_steps =('recovery_steps',  'mean'),
    max_lure_spike      =('lure_spike',      'max'),
).round(4).reset_index()
print(lure_summary.to_string(index=False))

# ── F. 정보 획득 효율성 집계 ─────────────────────────────────
print('\n--- 시나리오 F: 정보 획득 효율성 ---')
ig_df = pd.DataFrame(infogain_results)
print(ig_df[['model','density','mean_nll','mean_dx','corr_dx_nll']].to_string(index=False))

# dense vs sparse NLL 차이 (정보 밀도 감지력)
ig_pivot = ig_df.pivot_table(index='model', columns='density', values='mean_nll').reset_index()
ig_pivot['nll_density_diff'] = ig_pivot.get('sparse', 0) - ig_pivot.get('dense', 0)
print('\nNLL density diff (sparse - dense, 높을수록 밀도 구분 잘함):')
print(ig_pivot[['model','dense','sparse','nll_density_diff']].to_string(index=False))

# ── G. 장기 고착도 집계 ──────────────────────────────────────
print('\n--- 시나리오 G: 장기 문맥 고착도 ---')
fix_df = pd.DataFrame([{k:v for k,v in r.items() if k!='phi_trace'} for r in fixation_results])
fix_summary = fix_df.groupby('model').agg(
    mean_convergence =('convergence_step', 'mean'),
    mean_final_phi   =('final_phi_mean',   'mean'),
    mean_decay_rate  =('decay_rate',       'mean'),
).round(4).reset_index()
print(fix_summary.to_string(index=False))

# 저장
hybrid_df.to_csv(os.path.join(RUN_DIR, 'results_scenario_d.csv'), index=False, encoding='utf-8-sig')
lure_df.to_csv(os.path.join(RUN_DIR, 'results_scenario_e.csv'), index=False, encoding='utf-8-sig')
ig_df.to_csv(os.path.join(RUN_DIR, 'results_scenario_f.csv'), index=False, encoding='utf-8-sig')
fix_df.to_csv(os.path.join(RUN_DIR, 'results_scenario_g.csv'), index=False, encoding='utf-8-sig')

# PAPER.md용 추가 테이블
print('\n--- PAPER.md 반영용 (시나리오 E~G) ---')
print('| Model | Lure Spike | Recovery Steps | Corr(Δx,NLL) dense | Fixation Conv | Decay Rate |')
print('|---|---|---|---|---|---|')
for m in MODEL_ORDER:
    ls  = lure_summary[lure_summary['model']==m]['mean_lure_spike'].values
    rs  = lure_summary[lure_summary['model']==m]['mean_recovery_steps'].values
    cr  = ig_df[(ig_df['model']==m) & (ig_df['density']=='dense')]['corr_dx_nll'].values
    fc  = fix_summary[fix_summary['model']==m]['mean_convergence'].values
    dr  = fix_summary[fix_summary['model']==m]['mean_decay_rate'].values
    ls_v = ls[0]  if len(ls)  else float('nan')
    rs_v = rs[0]  if len(rs)  else float('nan')
    cr_v = cr[0]  if len(cr)  else float('nan')
    fc_v = fc[0]  if len(fc)  else float('nan')
    dr_v = dr[0]  if len(dr)  else float('nan')
    print(f'| {m:20s} | {ls_v:+.4f} | {rs_v:.1f} | {cr_v:+.3f} | {fc_v:.0f} | {dr_v:.4f} |')



§4.5 LLM EXPERIMENT — gemma4_26b  (12-Expert)

--- ΔH Summary ---
  BiLevel_MoE         : Stable H=0.1164  Trans H=0.4152  ΔH=+0.2988
  ECR                 : Stable H=0.3679  Trans H=1.2396  ΔH=+0.8717
  MoD                 : Stable H=0.1059  Trans H=0.4667  ΔH=+0.3608
  Nomadic_Full        : Stable H=0.1544  Trans H=0.4833  ΔH=+0.3289
  RoutingFree_MoE     : Stable H=0.3722  Trans H=1.1258  ΔH=+0.7536
  Sigma_MoE           : Stable H=0.1230  Trans H=0.4947  ΔH=+0.3717

--- Expert Load & Diversity (stable phase) ---
          model  mean_load_cv  mean_load_ent  mean_gini
    BiLevel_MoE        0.2629         0.6948    -0.3456
            ECR        1.1236         1.1962     0.3336
            MoD        0.7963         0.7168     0.0542
   Nomadic_Full        0.9259         0.9550     0.1875
RoutingFree_MoE        0.6008         2.2036     0.2187
      Sigma_MoE        1.0018         1.3225     0.3076

--- Transition Sharpness ---
          model  mean_settling  mean_osc  mean_sw
    B

In [31]:
# ============================================================
# STEP 12: Visualization (Separated into 4 Figures & Saved to Drive)
# ============================================================
COLORS = {
    'Nomadic_Full':    '#2C6FAC',
    'MoD':             '#E07B54',
    'ECR':             '#4CAF50',
    'Sigma_MoE':       '#9C27B0',
    'BiLevel_MoE':     '#FF9800',
    'RoutingFree_MoE': '#888888',
}

# Helper for bar labels
def add_labels(ax, bars, fmt='{:.3f}'):
    for b in bars:
        val = b.get_height()
        ax.text(b.get_x()+b.get_width()/2., val + abs(val)*0.01,
                fmt.format(val), ha='center', fontsize=8, fontweight='bold')

# --- FIG 1: Core Performance Metrics ---
fig1, axes1 = plt.subplots(1, 3, figsize=(18, 5))
dh_vals = []
for m in MODEL_ORDER:
    sh = summary[(summary['model']==m) & (summary['phase']=='stable')]['mean_H'].values[0]
    th = summary[(summary['model']==m) & (summary['phase']=='transition')]['mean_H'].values[0]
    dh_vals.append(th - sh)
bars = axes1[0].bar(MODEL_ORDER, dh_vals, color=[COLORS[m] for m in MODEL_ORDER])
axes1[0].set_title('Delta H (Trans H - Stable H)')
axes1[0].tick_params(axis='x', rotation=30)
add_labels(axes1[0], bars, '{:+.3f}')
for i, m in enumerate(MODEL_ORDER):
    vals = [df_a[(df_a['model']==m) & (df_a['domain']==d)]['perplexity'].mean() for d in DOMAINS]
    axes1[1].bar(np.arange(3) + i*0.12, vals, 0.12, label=m, color=COLORS[m])
axes1[1].set_xticks(np.arange(3) + 0.3); axes1[1].set_xticklabels(DOMAINS)
axes1[1].set_title('Perplexity by Domain')
lat_vals = [next((l['latency_ms'] for l in latency_results if l['model']==m), 0.0) for m in MODEL_ORDER]
bars = axes1[2].bar(MODEL_ORDER, lat_vals, color=[COLORS[m] for m in MODEL_ORDER])
axes1[2].set_title('Latency (ms/token)')
axes1[2].tick_params(axis='x', rotation=30)
add_labels(axes1[2], bars, '{:.1f}')
plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, 'fig1_core_performance.png'), dpi=150)
plt.show()

# --- FIG 2: Routing & Load Stats ---
fig2, axes2 = plt.subplots(1, 3, figsize=(18, 5))
vals_cv = [summary[(summary['model']==m) & (summary['phase']=='stable')]['mean_load_cv'].values[0] for m in MODEL_ORDER]
bars0 = axes2[0].bar(MODEL_ORDER, vals_cv, color=[COLORS[m] for m in MODEL_ORDER])
axes2[0].set_title('Expert Load CV')
add_labels(axes2[0], bars0)
vals_st = [summary[(summary['model']==m) & (summary['phase']=='stable')]['mean_settling'].values[0] for m in MODEL_ORDER]
bars1 = axes2[1].bar(MODEL_ORDER, vals_st, color=[COLORS[m] for m in MODEL_ORDER])
axes2[1].set_title('Settling Time (Steps)')
add_labels(axes2[1], bars1, '{:.2f}')
vals_osc = [summary[(summary['model']==m) & (summary['phase']=='stable')]['mean_osc'].values[0] for m in MODEL_ORDER]
bars2 = axes2[2].bar(MODEL_ORDER, vals_osc, color=[COLORS[m] for m in MODEL_ORDER])
axes2[2].set_title('Oscillation Rate')
add_labels(axes2[2], bars2)
plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, 'fig2_routing_stats.png'), dpi=150)
plt.show()

# --- FIG 3: Stress Scenarios B & C ---
fig3, axes3 = plt.subplots(1, 2, figsize=(14, 5))
pre = [shift_df[shift_df['model']==m]['pre_mean_dx'].values[0] for m in MODEL_ORDER]
post = [shift_df[shift_df['model']==m]['post_mean_dx'].values[0] for m in MODEL_ORDER]
x = np.arange(len(MODEL_ORDER))
axes3[0].bar(x - 0.2, pre, 0.4, label='Pre-Shift', color='steelblue')
axes3[0].bar(x + 0.2, post, 0.4, label='Post-Shift', color='tomato')
axes3[0].set_xticks(x); axes3[0].set_xticklabels(MODEL_ORDER, rotation=30)
axes3[0].set_title('Scenario B: Context Shift (Delta-x)')
axes3[0].legend()
clean = [noise_df[(noise_df['model']==m) & (noise_df['condition']=='clean')]['entropy'].values[0] for m in MODEL_ORDER]
noisy = [noise_df[(noise_df['model']==m) & (noise_df['condition']=='noisy')]['entropy'].values[0] for m in MODEL_ORDER]
axes3[1].bar(x - 0.2, clean, 0.4, label='Clean', color='seagreen')
axes3[1].bar(x + 0.2, noisy, 0.4, label='Noisy', color='darkorange')
axes3[1].set_xticks(x); axes3[1].set_xticklabels(MODEL_ORDER, rotation=30)
axes3[1].set_title('Scenario C: Noise Immunity (Entropy)')
axes3[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, 'fig3_stress_scenarios.png'), dpi=150)
plt.show()

# --- FIG 4: Long-Context & Recovery ---
fig4, axes4 = plt.subplots(1, 2, figsize=(14, 5))
spikes = [lure_summary[lure_summary['model']==m]['mean_lure_spike'].values[0] for m in MODEL_ORDER]
steps = [lure_summary[lure_summary['model']==m]['mean_recovery_steps'].values[0] for m in MODEL_ORDER]
ax_rec = axes4[0].twinx()
axes4[0].bar(x - 0.2, spikes, 0.4, color='tomato', label='Lure Spike')
ax_rec.bar(x + 0.2, steps, 0.4, color='steelblue', label='Recovery Steps')
axes4[0].set_xticks(x); axes4[0].set_xticklabels(MODEL_ORDER, rotation=30)
axes4[0].set_title('Scenario E: Lure Spike & Recovery Steps')
dom_colors = {'math': '#2C6FAC', 'code': '#E07B54', 'language': '#4CAF50'}
for row_f in [r for r in fixation_results if r['model'] == 'Nomadic_Full']:
    axes4[1].plot(row_f['phi_trace'], color=dom_colors[row_f['domain']], label=row_f['domain'])
axes4[1].axhline(0.02, color='red', linestyle='--', alpha=0.5, label='Threshold')
axes4[1].set_title('Scenario G: Long-Context Fixation (Nomadic Full)')
axes4[1].set_xlabel('Token Index'); axes4[1].set_ylabel('Delta-x')
axes4[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, 'fig4_recovery_fixation.png'), dpi=150)
plt.show()

print(f'✅ All figures saved to: {RUN_DIR}')

✅ All figures saved to: /content/drive/MyDrive/nomadic_llm_results/gemma4_26b_20260419_021120
